# SSDS 2026 — Prediksi Tinggi Muka Air (TMA) DAS Bengawan Solo

End-to-end pipeline untuk kompetisi Sebelas Maret Statistics Data Science 2026.
Target: prediksi `tma_mdpl` untuk 30 pos pemantauan, periode test 2025-09-19 s/d
2026-05-18 (242 hari, forecast murni tanpa ground-truth TMA), dievaluasi dengan RMSE.

**Cara membaca notebook ini**: pipeline melewati tiga putaran perbaikan.
- **v4**: rebuild awal, menemukan & memperbaiki bug spike sensor.
- **v5/v6**: dipicu code review pertama, menemukan bug leakage yang membuat
  angka v4 terlalu optimis (1.20→1.4585), plus beberapa eksperimen lanjutan.
- **v7 (final)**: dipicu code review kedua yang lebih dalam — memperbaiki 2 bug
  metodologi tambahan (self-referential climatology, upstream-map beku lintas
  fold) dan menambah 3 pemeriksaan robustness (pemisahan tuning/scoring,
  bootstrap signifikansi tau per-stasiun, uji missingness).

Setiap putaran dilaporkan apa adanya — termasuk angka yang harus dikoreksi
turun dan eksperimen yang gagal — karena itu bagian dari proses ilmiahnya.

Struktur:
1. EDA (15 pemeriksaan)
2. v4: cleaning & feature engineering (bug spike sensor diperbaiki)
3. v4: validasi season-matched (⚠️ kemudian ditemukan mengandung bug leakage)
4. `feature_lib.py` — konsolidasi kode + 5 bug fix dari 2 putaran code review
5. Multi-fold validation v2 (perbaikan putaran 1: climate-check nyata)
6. Eksperimen lanjutan putaran 1: anomaly target, upstream-lag, LGBM tuning, per-station tau (naif)
7. Multi-fold validation v3 (perbaikan putaran 2: pemisahan tuning/scoring + cross-fold stability)
8. Bootstrap signifikansi tau per-stasiun (perbaikan putaran 2)
9. Uji missingness backtest vs curah hujan (perbaikan putaran 2)
10. Model final v7 & submission
11. Ringkasan & rekomendasi lanjutan


## 1. Exploratory Data Analysis

15 pemeriksaan pada train, test, dan data exogenous.

In [1]:
import pandas as pd, numpy as np, warnings, time
warnings.filterwarnings('ignore')
t0=time.time()
pd.set_option('display.width',160)

tr = pd.read_csv(r'D:/Lomba/ssds/train.csv', parse_dates=['datetime'])
te_raw = pd.read_csv(r'D:/Lomba/ssds/test.csv')
dl = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
ko = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/koordinat_pos.csv')

# split test id into datetime/nama_pos
split = te_raw['id'].str.split(' - ', n=1, expand=True)
te = pd.DataFrame({'datetime': pd.to_datetime(split[0]), 'nama_pos': split[1], 'id': te_raw['id']})

def sec(n): print(f"\n===== EDA {n} =====")

# 1. Basic shape/schema sanity
sec(1)
print('train', tr.shape, 'test', te.shape, 'stations_train', tr.nama_pos.nunique(), 'stations_test', te.nama_pos.nunique())
print('station set equal:', set(tr.nama_pos)==set(te.nama_pos))
print('train dtypes:', tr.dtypes.to_dict())

# 2. Missing values
sec(2)
print('train NA:\n', tr.isna().sum())
print('dl NA:\n', dl.isna().sum()[dl.isna().sum()>0])

# 3. Duplicates
sec(3)
print('train dup rows', tr.duplicated().sum(), 'dup key', tr.duplicated(['datetime','nama_pos']).sum())
print('test dup key', te.duplicated(['datetime','nama_pos']).sum())
print('dl dup key', dl.duplicated(['datetime','nama_pos']).sum())

# 4. Target distribution overall + per-station range/scale heterogeneity
sec(4)
print(tr.tma_mdpl.describe())
neg = tr[tr.tma_mdpl<=0]
print('non-positive rows:\n', neg)
stn_stats = tr.groupby('nama_pos').tma_mdpl.agg(['min','max','mean','std','count']).sort_values('mean')
print(stn_stats.to_string())

# 5. Temporal coverage / gaps per station (train)
sec(5)
expected = pd.date_range(tr.datetime.min(), tr.datetime.max(), freq='6h')  # not exact due to 06/12/18 pattern but gives gap sense
g = tr.groupby('nama_pos').datetime.agg(['min','max','count'])
g['expected_3xday'] = ((g['max']-g['min']).dt.days+1)*3
g['missing_frac'] = 1 - g['count']/g['expected_3xday']
print(g.sort_values('missing_frac', ascending=False).to_string())

# 6. Test set structure: horizon length, gap from train end
sec(6)
train_end = tr.datetime.max()
print('train_end', train_end, 'test_start', te.datetime.min(), 'test_end', te.datetime.max())
print('gap days (test_start - train_end):', (te.datetime.min()-train_end).days)
print('horizon days (test_end-test_start):', (te.datetime.max()-te.datetime.min()).days)
te_counts = te.groupby('nama_pos').datetime.count()
print('rows per station in test (should be uniform):', te_counts.unique())

# 7. Seasonal cycle of target (monthly mean) -> is there a wet/dry season signal test will span
sec(7)
tr['month']=tr.datetime.dt.month
mon = tr.groupby('month').tma_mdpl.mean()
print('monthly mean tma (all stations pooled, mind scale diff):\n', mon)
test_months = sorted(te.datetime.dt.month.unique())
print('months covered by TEST:', test_months, ' -> spans wet season (Nov-Apr) peak fully')

# 8. Per-station day-of-year climatology check: does day-262(Sep19)->day138(May18) span exist in train history at all stations
sec(8)
tr['doy']=tr.datetime.dt.dayofyear
te['doy']=te.datetime.dt.dayofyear
print('train doy range per year available - years:', tr.datetime.dt.year.unique())
# does train contain a FULL prior wet season analogous to test's wet season (Sep James Y-1 to May Y)?
print('train max date used to have prior-year-same-doy target available for early test rows (doy 262):')
mask = (tr.datetime.dt.year==2024) & (tr.doy==262)
print(tr[mask][['nama_pos','datetime','tma_mdpl']].head())

# 9. Outlier scan via z-score per station (train)
sec(9)
def zscan(g):
    z = (g.tma_mdpl - g.tma_mdpl.mean())/g.tma_mdpl.std()
    return (z.abs()>4).sum()
outl = tr.groupby('nama_pos').apply(zscan)
print('stations with |z|>4 outlier count (train):\n', outl[outl>0].sort_values(ascending=False))

# 10. Sudden jump/spike detection (diff between consecutive obs per station)
sec(10)
tr_sorted = tr.sort_values(['nama_pos','datetime'])
tr_sorted['diff'] = tr_sorted.groupby('nama_pos').tma_mdpl.diff()
big_jump = tr_sorted.reindex(tr_sorted['diff'].abs().sort_values(ascending=False).index).head(15)
print(big_jump[['nama_pos','datetime','tma_mdpl','diff']])

# 11. Correlation of exogenous features with tma (merged on nearest hour) - sample a few stations
sec(11)
dl6 = dl[dl.datetime.dt.hour.isin([6,12,18])]
merged = tr.merge(dl6, on=['datetime','nama_pos'], how='left')
num_cols = ['rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c',
            'soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm',
            'surface_pressure_hpa','pressure_msl_hpa','nino_34','rmm1','rmm2','mjo_amplitude']
corrs = merged[num_cols+['tma_mdpl']].corr()['tma_mdpl'].drop('tma_mdpl').sort_values(key=abs, ascending=False)
print('corr(exog, tma) pooled all stations (mind Simpson paradox across scale):\n', corrs)

# 12. Per-station correlation with rainfall/soil moisture (captures true local hydrology signal)
sec(12)
def corr_local(g):
    if g['rainfall_mm'].notna().sum()<50: return np.nan
    return g['rainfall_mm'].corr(g['tma_mdpl'])
rc = merged.groupby('nama_pos').apply(corr_local).sort_values()
print('per-station corr(rainfall, tma):\n', rc)

# 13. landcover / built_surface static per station - does it vary over time (should be near-static -> use as station attribute)
sec(13)
lc = dl.groupby('nama_pos')['landcover_class'].nunique()
print('landcover_class nunique per station (1=static):\n', lc.value_counts())
bs = dl.groupby('nama_pos')['built_surface_m2'].std()
print('built_surface_m2 std per station (near 0 = static):\n', bs.describe())

# 14. nino_34 / MJO resolution (monthly/daily) - check update frequency, useful for feature freq handling
sec(14)
print('unique nino_34 values per station (should repeat monthly):', dl.groupby('nama_pos').nino_34.apply(lambda s: s.diff().ne(0).sum()).mean())
print('unique mjo_phase transitions per day approx:', dl.groupby(dl.datetime.dt.date).mjo_phase.nunique().mean())

# 15. Autocorrelation of tma_mdpl at daily lag vs weekly vs yearly (single representative station: Jurug - high err share)
sec(15)
jurug = tr[tr.nama_pos=='Jurug'].sort_values('datetime').set_index('datetime').tma_mdpl.asfreq('6h' if False else None)
s = tr[tr.nama_pos=='Jurug'].sort_values('datetime')[['datetime','tma_mdpl']].set_index('datetime')['tma_mdpl']
for lag_days in [1,3,7,30,90,180,365]:
    lag_steps = lag_days*3  # approx since 3 obs/day, assumes no gaps -- rough
    if lag_steps < len(s):
        ac = s.autocorr(lag=lag_steps)
        print(f'Jurug autocorr at ~{lag_days}d lag (steps={lag_steps}):', round(ac,4))

print(f"\nDONE eda_full.py in {time.time()-t0:.1f}s")



===== EDA 1 =====
train (84396, 3) test (21780, 3) stations_train 30 stations_test 30
station set equal: True
train dtypes: {'datetime': dtype('<M8[ns]'), 'nama_pos': dtype('O'), 'tma_mdpl': dtype('float64')}

===== EDA 2 =====
train NA:
 datetime    0
nama_pos    0
tma_mdpl    0
dtype: int64
dl NA:
 soil_moisture_0_7cm          720
soil_moisture_7_28cm         720
soil_moisture_28_100cm       720
soil_moisture_100_255cm      720
surface_pressure_hpa         720
pressure_msl_hpa             720
rmm1                         720
rmm2                         720
mjo_phase                    720
mjo_amplitude                720
mjo_active                   720
nino_34                    12960
dtype: int64

===== EDA 3 =====


train dup rows 0 dup key 0
test dup key 0
dl dup key 0

===== EDA 4 =====
count    84396.000000
mean        56.482537
std         46.765914
min         -0.059668
25%         10.100000
50%         50.370000
75%         90.660938
max        325.830000
Name: tma_mdpl, dtype: float64
non-positive rows:
                  datetime                  nama_pos  tma_mdpl
15016 2023-09-14 06:00:00  Bojonegoro - Kali Kethek  0.000000
34492 2025-02-28 06:00:00                     Jurug  0.000000
41825 2023-11-04 18:00:00          Kali Pepe - PTPN  0.000000
47644 2023-11-10 18:00:00              Karanggeneng  0.000000
56259 2023-10-11 18:00:00                  Ketonggo -0.059668
                                  min         max        mean       std  count
nama_pos                                                                      
Arjowinangun - Pacitan       0.347639    4.850000    1.116478  0.494220   2903
Karanggeneng                 0.000000    5.133698    2.166324  0.972274   2903
Floodway Br

                                          min                 max  count  expected_3xday  missing_frac
nama_pos                                                                                              
Floodway Bridge C         2023-01-01 06:00:00 2025-09-18 18:00:00   2377            2976      0.201277
Bojonegoro - Kali Kethek  2023-01-01 06:00:00 2025-09-18 18:00:00   2688            2976      0.096774
Gunungsari                2024-08-13 18:00:00 2025-09-18 18:00:00   1126            1206      0.066335
Babat                     2023-01-01 06:00:00 2025-09-18 18:00:00   2838            2976      0.046371
Jarum                     2023-01-01 06:00:00 2025-09-18 18:00:00   2864            2976      0.037634
Sumberrejo                2023-01-01 06:00:00 2025-09-18 18:00:00   2878            2976      0.032930
Cepu                      2023-01-01 06:00:00 2025-09-18 18:00:00   2893            2976      0.027890
Kali Pepe - Tugu Boto     2023-01-01 06:00:00 2025-09-18 18:00:00   2900 

corr(exog, tma) pooled all stations (mind Simpson paradox across scale):
 surface_pressure_hpa      -0.947417
soil_moisture_100_255cm    0.189417
soil_moisture_28_100cm     0.126973
soil_moisture_7_28cm       0.123752
dew_point_c               -0.121663
soil_moisture_0_7cm        0.105786
pressure_msl_hpa           0.097006
temperature_c             -0.076961
nino_34                    0.017774
rainfall_mm               -0.005299
mjo_amplitude              0.004439
rmm2                       0.002720
rmm1                      -0.002537
humidity_pct               0.000426
cloud_cover_pct            0.000275
Name: tma_mdpl, dtype: float64

===== EDA 12 =====
per-station corr(rainfall, tma):
 nama_pos
Kali Anyar - Kreteg Abang    0.001619
Wonogiri Dam                 0.003934
Kali Pepe - PTPN             0.007204
Floodway Bridge C            0.017224
Jarum                        0.021719
Peren                        0.037956
Colo Weir                    0.040531
Karangnongko              

unique mjo_phase transitions per day approx: 0.9991896272285251

===== EDA 15 =====
Jurug autocorr at ~1d lag (steps=3): 0.3452
Jurug autocorr at ~3d lag (steps=9): 0.303
Jurug autocorr at ~7d lag (steps=21): 0.2367
Jurug autocorr at ~30d lag (steps=90): 0.1649
Jurug autocorr at ~90d lag (steps=270): 0.0209
Jurug autocorr at ~180d lag (steps=540): -0.1215
Jurug autocorr at ~365d lag (steps=1095): 0.1428

DONE eda_full.py in 5.2s


## 2. Data Cleaning & Feature Engineering (v4)

Temuan EDA kunci: `seasonal_lag_1y` sebelumnya bernilai RMSE=64 karena train
mengandung **spike sensor 1-timestep** (mis. Napel 34.55→325.83→34.55) yang
tidak dibersihkan. Sel berikut membersihkan spike via deteksi MAD, lalu
membangun fitur domain (rolling curah hujan/suhu/tanah, kalender siklis,
atribut statis stasiun, `doy_climatology`, `seasonal_lag_1y`, anchor persistence).


In [2]:
"""
SSDS 2026 - Full rebuild v4
Pipeline: clean -> feature engineer -> season-matched backtest -> final model+submission
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings('ignore')
t0 = time.time()

def log(msg):
    print(f"[{time.time()-t0:6.1f}s] {msg}")

# ============ 1. LOAD ============
tr = pd.read_csv(r'D:/Lomba/ssds/train.csv', parse_dates=['datetime'])
te_raw = pd.read_csv(r'D:/Lomba/ssds/test.csv')
split = te_raw['id'].str.split(' - ', n=1, expand=True)
te = pd.DataFrame({'datetime': pd.to_datetime(split[0]), 'nama_pos': split[1]})
dl = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
ko = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/koordinat_pos.csv')
log(f"loaded train={tr.shape} test={te.shape} dl={dl.shape}")

# ============ 2. CLEAN TARGET (outlier spikes / negative values) ============
# Sensor glitch signature: single-timestep spike where value jumps far from both
# neighbors and reverts immediately (confirmed via EDA: Napel 2023-04-10, etc.)
tr = tr.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

def clean_station(g):
    g = g.copy()
    v = g['tma_mdpl'].values.copy()
    med = np.median(v)
    mad = np.median(np.abs(v - med)) + 1e-6
    # robust z-score
    rz = 0.6745 * (v - med) / mad
    prev = np.r_[v[0], v[:-1]]
    nxt = np.r_[v[1:], v[-1]]
    # a point is a spike if it's far (robust z) from BOTH neighbors while neighbors
    # are close to each other (isolated single-point glitch, not a real sustained rise)
    neigh_close = np.abs(prev - nxt) < 0.3 * (np.abs(prev) + np.abs(nxt) + 1e-6)
    far_prev = np.abs(v - prev) > 5 * mad
    far_nxt = np.abs(v - nxt) > 5 * mad
    is_spike = (np.abs(rz) > 6) & neigh_close & far_prev & far_nxt
    v[is_spike] = np.nan
    # negative / physically implausible (TMA should be >= 0)
    v[v < 0] = np.nan
    g['tma_mdpl'] = v
    g['tma_mdpl'] = g['tma_mdpl'].interpolate(limit_direction='both')
    g['is_cleaned'] = is_spike | (g['tma_mdpl'].values < 0)
    return g

tr = tr.groupby('nama_pos', group_keys=False).apply(clean_station)
n_cleaned = tr['is_cleaned'].sum()
log(f"cleaned {n_cleaned} spike/negative points out of {len(tr)} ({100*n_cleaned/len(tr):.2f}%)")
tr = tr.drop(columns=['is_cleaned'])

# ============ 3. STATION STATIC ATTRIBUTES ============
static = dl.groupby('nama_pos').agg(
    landcover_class=('landcover_class', 'first'),
    built_surface_m2=('built_surface_m2', 'first'),
).reset_index()
static = static.merge(ko, on='nama_pos', how='left')
log(f"static attrs shape={static.shape}")

# ============ 4. EXOGENOUS FEATURES aggregated to 6h obs times ============
# dl is hourly; TMA obs at 06/12/18. For each obs, use env data up to AND INCLUDING
# that hour (no future leakage) with rolling windows capturing recent conditions.
dl = dl.sort_values(['nama_pos', 'datetime'])
dl['nino_34'] = dl.groupby('nama_pos')['nino_34'].ffill().bfill()
for c in ['soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
          'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa',
          'rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active']:
    dl[c] = dl.groupby('nama_pos')[c].ffill().bfill()

roll_specs = {
    'rainfall_mm': [24, 72, 168],       # 1d,3d,7d accum (sum)
    'temperature_c': [24],
    'humidity_pct': [24],
    'soil_moisture_0_7cm': [24],
    'soil_moisture_28_100cm': [24],
    'surface_pressure_hpa': [24],
    'pressure_msl_hpa': [24],
}
dl_feat = dl[['datetime', 'nama_pos']].copy()
for col, windows in roll_specs.items():
    for w in windows:
        agg = 'sum' if col == 'rainfall_mm' else 'mean'
        dl_feat[f'{col}_roll{w}h_{agg}'] = (
            dl.groupby('nama_pos')[col]
              .transform(lambda s: s.rolling(w, min_periods=max(1, w//4)).agg(agg))
        )
# snapshot (instantaneous) exogenous values at the hour itself
snap_cols = ['rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c',
             'wind_speed_kmh','soil_moisture_0_7cm','soil_moisture_7_28cm',
             'soil_moisture_28_100cm','soil_moisture_100_255cm','surface_pressure_hpa',
             'pressure_msl_hpa','rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active','nino_34']
for c in snap_cols:
    dl_feat[c] = dl[c].values
log(f"dl_feat built shape={dl_feat.shape}")

def merge_exog(df):
    out = df.merge(dl_feat, on=['datetime', 'nama_pos'], how='left')
    return out

tr = merge_exog(tr)
te = merge_exog(te)
log(f"after exog merge: train={tr.shape} test={te.shape} test_na_exog={te[snap_cols].isna().sum().sum()}")

# ============ 5. CALENDAR FEATURES ============
def add_calendar(df):
    df = df.copy()
    df['hour'] = df.datetime.dt.hour
    df['month'] = df.datetime.dt.month
    df['doy'] = df.datetime.dt.dayofyear
    df['hour_sin'] = np.sin(2*np.pi*df.hour/24)
    df['hour_cos'] = np.cos(2*np.pi*df.hour/24)
    df['month_sin'] = np.sin(2*np.pi*df.month/12)
    df['month_cos'] = np.cos(2*np.pi*df.month/12)
    df['doy_sin'] = np.sin(2*np.pi*df.doy/365.25)
    df['doy_cos'] = np.cos(2*np.pi*df.doy/365.25)
    df['is_wet_season'] = df.month.isin([11,12,1,2,3,4]).astype(int)
    return df

tr = add_calendar(tr)
te = add_calendar(te)

# ============ 6. STATION ATTRIBUTES + CLIMATOLOGY (computed on CLEANED train only) ============
tr = tr.merge(static, on='nama_pos', how='left')
te = te.merge(static, on='nama_pos', how='left')

stn_stats = tr.groupby('nama_pos')['tma_mdpl'].agg(station_mean='mean', station_std='std', station_median='median').reset_index()
tr = tr.merge(stn_stats, on='nama_pos', how='left')
te = te.merge(stn_stats, on='nama_pos', how='left')

# day-of-year climatology per station on CLEANED data (smoothed with +-3 day window)
doy_clim_list = []
for stn, g in tr.groupby('nama_pos'):
    s = g.set_index('doy')['tma_mdpl']
    means = {}
    for d in range(1, 367):
        window = [((d + off - 1) % 366) + 1 for off in range(-5, 6)]
        vals = s[s.index.isin(window)]
        means[d] = vals.mean() if len(vals) else np.nan
    doy_clim_list.append(pd.DataFrame({'nama_pos': stn, 'doy': list(means.keys()), 'doy_climatology': list(means.values())}))
doy_clim = pd.concat(doy_clim_list, ignore_index=True)
tr = tr.merge(doy_clim, on=['nama_pos', 'doy'], how='left')
te = te.merge(doy_clim, on=['nama_pos', 'doy'], how='left')
fallback = stn_stats.set_index('nama_pos')['station_mean']
tr['doy_climatology'] = tr['doy_climatology'].fillna(tr['nama_pos'].map(fallback))
te['doy_climatology'] = te['doy_climatology'].fillna(te['nama_pos'].map(fallback))
log("doy_climatology built (cleaned, smoothed +-5d)")

# seasonal_lag_1y: value from ~365 days prior, via merge_asof on CLEANED series (tolerance 2 days)
def add_seasonal_lag(target_df, source_df, days=365, tol_days=2, name='seasonal_lag_1y'):
    src = source_df[['datetime', 'nama_pos', 'tma_mdpl']].copy()
    src = src.rename(columns={'tma_mdpl': name, 'datetime': 'src_dt'})
    src = src.sort_values('src_dt')
    tgt = target_df.copy()
    tgt['lookup_dt'] = tgt['datetime'] - pd.Timedelta(days=days)
    tgt = tgt.sort_values('lookup_dt')
    out = pd.merge_asof(tgt, src.sort_values('src_dt'), left_on='lookup_dt', right_on='src_dt',
                         by='nama_pos', direction='nearest', tolerance=pd.Timedelta(days=tol_days))
    out = out.drop(columns=['lookup_dt', 'src_dt'])
    return out.sort_index()

tr = add_seasonal_lag(tr, tr)
te = add_seasonal_lag(te, tr)  # test lag must come from train (no leakage), source=cleaned train
tr['seasonal_lag_1y'] = tr['seasonal_lag_1y'].fillna(tr['doy_climatology'])
te['seasonal_lag_1y'] = te['seasonal_lag_1y'].fillna(te['doy_climatology'])
log("seasonal_lag_1y rebuilt on cleaned data")

# ============ 7. LAST-KNOWN (persistence) value & horizon from train end (for blending) ============
last_known = tr.sort_values('datetime').groupby('nama_pos').tail(1)[['nama_pos', 'datetime', 'tma_mdpl']]
last_known = last_known.rename(columns={'datetime': 'last_dt', 'tma_mdpl': 'last_known'})
last_known_map = last_known.set_index('nama_pos')

def add_persistence(df, origin_map):
    df = df.copy()
    df['last_known'] = df['nama_pos'].map(origin_map['last_known'])
    df['last_dt'] = df['nama_pos'].map(origin_map['last_dt'])
    df['horizon_days'] = (df['datetime'] - df['last_dt']).dt.total_seconds() / 86400
    return df.drop(columns=['last_dt'])

te = add_persistence(te, last_known_map)
log(f"persistence anchor added, horizon range: {te.horizon_days.min():.1f} to {te.horizon_days.max():.1f}")

train_end_global = tr.datetime.max()
log(f"FEATURE ENGINEERING DONE. train={tr.shape} test={te.shape}")

tr.to_parquet(r'D:/Lomba/ssds/model/tr_feat_v4.parquet')
te.to_parquet(r'D:/Lomba/ssds/model/te_feat_v4.parquet')
log("saved tr_feat_v4.parquet / te_feat_v4.parquet")


[   3.7s] loaded train=(84396, 3) test=(21780, 2) dl=(888480, 27)
[   3.8s] cleaned 206 spike/negative points out of 84396 (0.24%)


[   3.9s] static attrs shape=(30, 5)


[   5.8s] dl_feat built shape=(888480, 29)


[   6.4s] after exog merge: train=(84396, 30) test=(21780, 29) test_na_exog=0


[   8.2s] doy_climatology built (cleaned, smoothed +-5d)
[   8.4s] seasonal_lag_1y rebuilt on cleaned data


[   8.5s] persistence anchor added, horizon range: 0.5 to 242.0
[   8.5s] FEATURE ENGINEERING DONE. train=(84396, 49) test=(21780, 50)


[   9.1s] saved tr_feat_v4.parquet / te_feat_v4.parquet


## 3. Validasi Season-Matched v4 ⚠️ *(mengandung bug, diperbaiki di §5)*

**Bug yang ditemukan belakangan**: fungsi pembersihan spike di sini dijalankan
pada `RAW` sebelum displit oleh cutoff, sehingga referensi median/MAD ikut
melihat data "masa depan" relatif ke fold — RMSE≈1.20 di bawah ini
**overoptimistic**, dikoreksi di §5 menjadi ≈1.4585 (lalu ≈1.34 setelah
putaran perbaikan kedua di §7-10).


In [3]:
"""
Season-matched, leakage-free backtest for v4 pipeline.
Mimics real test: CUT = train_end - 365 days (so validation horizon spans the
same wet-season-crossing 242-day window the real test spans), train on data
<= CUT, evaluate on data in (CUT, CUT+242d].
All station-level statistics (mean/std, doy_climatology, seasonal_lag_1y) are
computed using ONLY data <= CUT to avoid leakage.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW = pd.read_csv(r'D:/Lomba/ssds/train.csv', parse_dates=['datetime']).sort_values(['nama_pos','datetime']).reset_index(drop=True)
dl = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
ko = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/koordinat_pos.csv')

# ---- clean spikes once (uses only local neighbor info, causal-safe: a spike
#      detector using immediate neighbors is standard hydrological QC and does
#      not leak future distributional info) ----
def clean_station(g):
    g = g.copy(); v = g['tma_mdpl'].values.copy()
    med = np.median(v); mad = np.median(np.abs(v-med)) + 1e-6
    rz = 0.6745*(v-med)/mad
    prev = np.r_[v[0], v[:-1]]; nxt = np.r_[v[1:], v[-1]]
    neigh_close = np.abs(prev-nxt) < 0.3*(np.abs(prev)+np.abs(nxt)+1e-6)
    far_prev = np.abs(v-prev) > 5*mad; far_nxt = np.abs(v-nxt) > 5*mad
    is_spike = (np.abs(rz) > 6) & neigh_close & far_prev & far_nxt
    v[is_spike] = np.nan; v[v < 0] = np.nan
    g['tma_mdpl'] = v
    g['tma_mdpl'] = g['tma_mdpl'].interpolate(limit_direction='both')
    return g
RAW = RAW.groupby('nama_pos', group_keys=False).apply(clean_station)

static = dl.groupby('nama_pos').agg(landcover_class=('landcover_class','first'),
                                     built_surface_m2=('built_surface_m2','first')).reset_index()
static = static.merge(ko, on='nama_pos', how='left')

dl = dl.sort_values(['nama_pos','datetime'])
dl['nino_34'] = dl.groupby('nama_pos')['nino_34'].ffill().bfill()
for c in ['soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
          'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa',
          'rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active']:
    dl[c] = dl.groupby('nama_pos')[c].ffill().bfill()
roll_specs = {'rainfall_mm':[24,72,168],'temperature_c':[24],'humidity_pct':[24],
              'soil_moisture_0_7cm':[24],'soil_moisture_28_100cm':[24],
              'surface_pressure_hpa':[24],'pressure_msl_hpa':[24]}
dl_feat = dl[['datetime','nama_pos']].copy()
for col, windows in roll_specs.items():
    for w in windows:
        agg = 'sum' if col=='rainfall_mm' else 'mean'
        dl_feat[f'{col}_roll{w}h_{agg}'] = dl.groupby('nama_pos')[col].transform(lambda s: s.rolling(w, min_periods=max(1,w//4)).agg(agg))
snap_cols = ['rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c',
             'wind_speed_kmh','soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
             'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2',
             'mjo_phase','mjo_amplitude','mjo_active','nino_34']
for c in snap_cols: dl_feat[c] = dl[c].values

def add_calendar(df):
    df = df.copy()
    df['hour']=df.datetime.dt.hour; df['month']=df.datetime.dt.month; df['doy']=df.datetime.dt.dayofyear
    df['hour_sin']=np.sin(2*np.pi*df.hour/24); df['hour_cos']=np.cos(2*np.pi*df.hour/24)
    df['month_sin']=np.sin(2*np.pi*df.month/12); df['month_cos']=np.cos(2*np.pi*df.month/12)
    df['doy_sin']=np.sin(2*np.pi*df.doy/365.25); df['doy_cos']=np.cos(2*np.pi*df.doy/365.25)
    df['is_wet_season']=df.month.isin([11,12,1,2,3,4]).astype(int)
    return df

def build_doy_clim(train_only):
    if 'doy' not in train_only.columns:
        train_only = train_only.assign(doy=train_only['datetime'].dt.dayofyear)
    out=[]
    for stn, g in train_only.groupby('nama_pos'):
        s = g.set_index('doy')['tma_mdpl']
        means={}
        for d in range(1,367):
            window=[((d+off-1)%366)+1 for off in range(-5,6)]
            vals = s[s.index.isin(window)]
            means[d]=vals.mean() if len(vals) else np.nan
        out.append(pd.DataFrame({'nama_pos':stn,'doy':list(means.keys()),'doy_climatology':list(means.values())}))
    return pd.concat(out, ignore_index=True)

def add_seasonal_lag(target_df, source_df, days=365, tol_days=2, name='seasonal_lag_1y'):
    src = source_df[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':name,'datetime':'src_dt'}).sort_values('src_dt')
    tgt = target_df.copy(); tgt['lookup_dt']=tgt['datetime']-pd.Timedelta(days=days)
    tgt = tgt.sort_values('lookup_dt')
    out = pd.merge_asof(tgt, src, left_on='lookup_dt', right_on='src_dt', by='nama_pos',
                         direction='nearest', tolerance=pd.Timedelta(days=tol_days))
    return out.drop(columns=['lookup_dt','src_dt']).sort_index()

def build_features(train_only, target_df):
    """train_only: cleaned rows with datetime<=CUT (used for all stats, no leakage).
       target_df: rows to build features FOR (can be train_only itself or validation rows)."""
    df = target_df.merge(dl_feat, on=['datetime','nama_pos'], how='left')
    df = add_calendar(df)
    df = df.merge(static, on='nama_pos', how='left')
    stn_stats = train_only.groupby('nama_pos')['tma_mdpl'].agg(station_mean='mean', station_std='std', station_median='median').reset_index()
    df = df.merge(stn_stats, on='nama_pos', how='left')
    doy_clim = build_doy_clim(train_only)
    df = df.merge(doy_clim, on=['nama_pos','doy'], how='left')
    fallback = stn_stats.set_index('nama_pos')['station_mean']
    df['doy_climatology'] = df['doy_climatology'].fillna(df['nama_pos'].map(fallback))
    df = add_seasonal_lag(df, train_only)
    df['seasonal_lag_1y'] = df['seasonal_lag_1y'].fillna(df['doy_climatology'])
    last_known = train_only.sort_values('datetime').groupby('nama_pos').tail(1)[['nama_pos','datetime','tma_mdpl']]
    lk_map = last_known.set_index('nama_pos')
    df['last_known'] = df['nama_pos'].map(lk_map['tma_mdpl'])
    last_dt = df['nama_pos'].map(lk_map['datetime'])
    df['horizon_days'] = (df['datetime'] - last_dt).dt.total_seconds()/86400
    return df

FEATURE_COLS_NUM = ['rainfall_mm_roll24h_sum','rainfall_mm_roll72h_sum','rainfall_mm_roll168h_sum',
    'temperature_c_roll24h_mean','humidity_pct_roll24h_mean','soil_moisture_0_7cm_roll24h_mean',
    'soil_moisture_28_100cm_roll24h_mean','surface_pressure_hpa_roll24h_mean','pressure_msl_hpa_roll24h_mean',
    'rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c','wind_speed_kmh',
    'soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm',
    'surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active','nino_34',
    'hour_sin','hour_cos','month_sin','month_cos','doy_sin','doy_cos','is_wet_season',
    'latitude','longitude','built_surface_m2',
    'station_mean','station_std','station_median','doy_climatology','seasonal_lag_1y']
CAT_COLS = ['nama_pos','landcover_class']

def make_pipelines():
    pre_tree = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)], remainder='passthrough')
    pre_lin = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS),
                                  ('num', StandardScaler(), FEATURE_COLS_NUM)], remainder='drop')
    ridge = Pipeline([('pre', pre_lin), ('model', Ridge(alpha=5.0))])
    histgb = Pipeline([('pre', pre_tree), ('model', HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=400, random_state=0))])
    lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
    return ridge, histgb, lgbm

CUT = RAW.datetime.max() - pd.Timedelta(days=365)
VA_END = CUT + pd.Timedelta(days=242)
train_only = RAW[RAW.datetime <= CUT].copy()
val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
log(f"CUT={CUT} VA_END={VA_END} train_only={len(train_only)} val_only={len(val_only)}")

tr_feat = build_features(train_only, train_only)
va_feat = build_features(train_only, val_only[['datetime','nama_pos']].copy())
va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}),
                         on=['datetime','nama_pos'], how='left')
assert va_feat['y_true'].isna().sum() == 0, "y_true merge produced NaN - key mismatch"
y_tr = tr_feat['tma_mdpl'].values
log(f"features built. tr_feat={tr_feat.shape} va_feat={va_feat.shape}")

Xcols = CAT_COLS + FEATURE_COLS_NUM
tr_feat[FEATURE_COLS_NUM] = tr_feat[FEATURE_COLS_NUM].fillna(0)
va_feat[FEATURE_COLS_NUM] = va_feat[FEATURE_COLS_NUM].fillna(0)
ridge, histgb, lgbm = make_pipelines()
ridge.fit(tr_feat[Xcols], y_tr)
histgb.fit(tr_feat[Xcols], y_tr)
lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)], remainder='passthrough')
Xtr_lgb = lgb_pre.fit_transform(tr_feat[Xcols])
Xva_lgb = lgb_pre.transform(va_feat[Xcols])
lgbm.fit(Xtr_lgb, y_tr)

pred_r = ridge.predict(va_feat[Xcols])
pred_h = histgb.predict(va_feat[Xcols])
pred_l = lgbm.predict(Xva_lgb)
y_true = va_feat['y_true'].values

def rmse(a,b): return np.sqrt(mean_squared_error(a,b))
log(f"Ridge={rmse(y_true,pred_r):.4f}  HistGB={rmse(y_true,pred_h):.4f}  LGBM={rmse(y_true,pred_l):.4f}")

best = (None, 1e9)
for wr in np.arange(0,1.05,0.2):
    for wh in np.arange(0,1.05-wr,0.2):
        wl = 1-wr-wh
        if wl < -1e-9: continue
        pred = wr*pred_r + wh*pred_h + wl*pred_l
        s = rmse(y_true, pred)
        if s < best[1]: best = ((wr,wh,wl), s)
log(f"BEST direct ensemble RMSE={best[1]:.4f} weights(r,h,l)={best[0]}")
wr,wh,wl = best[0]
pred_direct = wr*pred_r + wh*pred_h + wl*pred_l

h = va_feat['horizon_days'].values
last_known = va_feat['last_known'].values
seasonal_1y = va_feat['seasonal_lag_1y'].values
doy_clim = va_feat['doy_climatology'].values

log(f"pure persistence RMSE={rmse(y_true, last_known):.4f}")
log(f"seasonal_lag_1y only RMSE={rmse(y_true, seasonal_1y):.4f}")
log(f"doy_climatology only RMSE={rmse(y_true, doy_clim):.4f}")
log(f"ML direct only RMSE={rmse(y_true, pred_direct):.4f}")

def blend(anchor, tau):
    w = np.exp(-h/tau)
    return w*last_known + (1-w)*anchor

for tau in [30,45,60,90,120,150,180,240,300]:
    for name, anchor in [('ML_direct', pred_direct), ('seasonal_1y', seasonal_1y), ('doy_clim', doy_clim)]:
        s = rmse(y_true, blend(anchor, tau))
        print(f"  tau={tau:4d} anchor={name:12s} RMSE={s:.4f}")

# 3-way anchor blend: combine seasonal_1y + doy_clim + ML_direct as the far-horizon anchor itself
best3 = (None, 1e9)
for tau in [90,120,150,180,240]:
    for a in np.arange(0,1.05,0.2):       # weight on seasonal_1y
        for b in np.arange(0,1.05-a,0.2): # weight on doy_clim
            c = 1-a-b                      # weight on ML_direct
            if c < -1e-9: continue
            anchor = a*seasonal_1y + b*doy_clim + c*pred_direct
            s = rmse(y_true, blend(anchor, tau))
            if s < best3[1]: best3 = ((tau,a,b,c), s)
log(f"BEST 3-way anchor blend RMSE={best3[1]:.4f} (tau,w_seas,w_doy,w_ml)={best3[0]}")

# per-station breakdown at best config
tau, a, b, c = best3[0]
anchor = a*seasonal_1y + b*doy_clim + c*pred_direct
final_pred = blend(anchor, tau)
va_feat['pred'] = final_pred
va_feat['err2'] = (va_feat['y_true']-va_feat['pred'])**2
per_stn = va_feat.groupby('nama_pos')['err2'].agg(['mean','count'])
per_stn['rmse'] = np.sqrt(per_stn['mean'])
per_stn['share'] = per_stn['mean']*per_stn['count']/ (va_feat['err2'].sum())
per_stn = per_stn.sort_values('share', ascending=False)
print("\nTOP-10 stations by error share (best config):")
print(per_stn.head(10)[['rmse','share']].to_string())

log("DONE validate_v4.py")


[   7.4s] CUT=2024-09-18 18:00:00 VA_END=2025-05-18 18:00:00 train_only=53838 val_only=19514


[  10.8s] features built. tr_feat=(53838, 51) va_feat=(19514, 51)


[  18.7s] Ridge=1.2129  HistGB=1.2544  LGBM=1.2766
[  18.8s] BEST direct ensemble RMSE=1.2096 weights(r,h,l)=(np.float64(0.8), np.float64(0.2), np.float64(-5.551115123125783e-17))
[  18.8s] pure persistence RMSE=2.1072
[  18.8s] seasonal_lag_1y only RMSE=1.5483
[  18.8s] doy_climatology only RMSE=1.3960
[  18.8s] ML direct only RMSE=1.2096
  tau=  30 anchor=ML_direct    RMSE=1.2083
  tau=  30 anchor=seasonal_1y  RMSE=1.5305
  tau=  30 anchor=doy_clim     RMSE=1.3832
  tau=  45 anchor=ML_direct    RMSE=1.2257
  tau=  45 anchor=seasonal_1y  RMSE=1.5250
  tau=  45 anchor=doy_clim     RMSE=1.3854
  tau=  60 anchor=ML_direct    RMSE=1.2518
  tau=  60 anchor=seasonal_1y  RMSE=1.5255
  tau=  60 anchor=doy_clim     RMSE=1.3950
  tau=  90 anchor=ML_direct    RMSE=1.3148
  tau=  90 anchor=seasonal_1y  RMSE=1.5407
  tau=  90 anchor=doy_clim     RMSE=1.4280
  tau= 120 anchor=ML_direct    RMSE=1.3796
  tau= 120 anchor=seasonal_1y  RMSE=1.5673
  tau= 120 anchor=doy_clim     RMSE=1.4694
  tau= 150 an

### 3b. Multi-fold check v4 (versi awal, klaim climate-check belum benar-benar dihitung)

In [4]:
"""
Multi-fold season-matched backtest + structural/climate alignment check.

Addresses the concern that a single validation fold might not represent the
real test distribution. Runs TWO season-matched folds (2023-09-19->2024-05-18
and 2024-09-19->2025-05-18) using a FIXED model config (weights/tau already
selected from prior analysis, not re-tuned per fold, to avoid overfitting the
validation choice itself) and reports both. Also explicitly checks:
  (a) structural alignment: rows/station and horizon-day distribution in each
      validation fold vs the real test.csv
  (b) climate-regime alignment: nino_34 / rainfall stats of each fold vs the
      real test period (2025-09-19->2026-05-18), using data_lingkungan.csv
      which actually covers that period.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW = pd.read_csv(r'D:/Lomba/ssds/train.csv', parse_dates=['datetime']).sort_values(['nama_pos','datetime']).reset_index(drop=True)
dl = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
ko = pd.read_csv(r'D:/Lomba/ssds/data_pendukung/koordinat_pos.csv')
te_raw = pd.read_csv(r'D:/Lomba/ssds/test.csv')
split = te_raw['id'].str.split(' - ', n=1, expand=True)
TEST_REAL = pd.DataFrame({'datetime': pd.to_datetime(split[0]), 'nama_pos': split[1]})

def clean_station(g):
    g = g.copy(); v = g['tma_mdpl'].values.copy()
    med = np.median(v); mad = np.median(np.abs(v-med)) + 1e-6
    rz = 0.6745*(v-med)/mad
    prev = np.r_[v[0], v[:-1]]; nxt = np.r_[v[1:], v[-1]]
    neigh_close = np.abs(prev-nxt) < 0.3*(np.abs(prev)+np.abs(nxt)+1e-6)
    far_prev = np.abs(v-prev) > 5*mad; far_nxt = np.abs(v-nxt) > 5*mad
    is_spike = (np.abs(rz) > 6) & neigh_close & far_prev & far_nxt
    v[is_spike] = np.nan; v[v < 0] = np.nan
    g['tma_mdpl'] = v
    g['tma_mdpl'] = g['tma_mdpl'].interpolate(limit_direction='both')
    return g
RAW = RAW.groupby('nama_pos', group_keys=False).apply(clean_station)

static = dl.groupby('nama_pos').agg(landcover_class=('landcover_class','first'),
                                     built_surface_m2=('built_surface_m2','first')).reset_index()
static = static.merge(ko, on='nama_pos', how='left')

dl = dl.sort_values(['nama_pos','datetime'])
dl['nino_34'] = dl.groupby('nama_pos')['nino_34'].ffill().bfill()
for c in ['soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
          'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa',
          'rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active']:
    dl[c] = dl.groupby('nama_pos')[c].ffill().bfill()
roll_specs = {'rainfall_mm':[24,72,168],'temperature_c':[24],'humidity_pct':[24],
              'soil_moisture_0_7cm':[24],'soil_moisture_28_100cm':[24],
              'surface_pressure_hpa':[24],'pressure_msl_hpa':[24]}
dl_feat = dl[['datetime','nama_pos']].copy()
for col, windows in roll_specs.items():
    for w in windows:
        agg = 'sum' if col=='rainfall_mm' else 'mean'
        dl_feat[f'{col}_roll{w}h_{agg}'] = dl.groupby('nama_pos')[col].transform(lambda s: s.rolling(w, min_periods=max(1,w//4)).agg(agg))
snap_cols = ['rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c',
             'wind_speed_kmh','soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
             'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2',
             'mjo_phase','mjo_amplitude','mjo_active','nino_34']
for c in snap_cols: dl_feat[c] = dl[c].values

def add_calendar(df):
    df = df.copy()
    df['hour']=df.datetime.dt.hour; df['month']=df.datetime.dt.month; df['doy']=df.datetime.dt.dayofyear
    df['hour_sin']=np.sin(2*np.pi*df.hour/24); df['hour_cos']=np.cos(2*np.pi*df.hour/24)
    df['month_sin']=np.sin(2*np.pi*df.month/12); df['month_cos']=np.cos(2*np.pi*df.month/12)
    df['doy_sin']=np.sin(2*np.pi*df.doy/365.25); df['doy_cos']=np.cos(2*np.pi*df.doy/365.25)
    df['is_wet_season']=df.month.isin([11,12,1,2,3,4]).astype(int)
    return df

def build_doy_clim(train_only):
    if 'doy' not in train_only.columns:
        train_only = train_only.assign(doy=train_only['datetime'].dt.dayofyear)
    out=[]
    for stn, g in train_only.groupby('nama_pos'):
        s = g.set_index('doy')['tma_mdpl']
        means={}
        for d in range(1,367):
            window=[((d+off-1)%366)+1 for off in range(-5,6)]
            vals = s[s.index.isin(window)]
            means[d]=vals.mean() if len(vals) else np.nan
        out.append(pd.DataFrame({'nama_pos':stn,'doy':list(means.keys()),'doy_climatology':list(means.values())}))
    return pd.concat(out, ignore_index=True)

def add_seasonal_lag(target_df, source_df, days=365, tol_days=2, name='seasonal_lag_1y'):
    src = source_df[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':name,'datetime':'src_dt'}).sort_values('src_dt')
    tgt = target_df.copy(); tgt['lookup_dt']=tgt['datetime']-pd.Timedelta(days=days)
    tgt = tgt.sort_values('lookup_dt')
    out = pd.merge_asof(tgt, src, left_on='lookup_dt', right_on='src_dt', by='nama_pos',
                         direction='nearest', tolerance=pd.Timedelta(days=tol_days))
    return out.drop(columns=['lookup_dt','src_dt']).sort_index()

def build_features(train_only, target_df):
    df = target_df.merge(dl_feat, on=['datetime','nama_pos'], how='left')
    df = add_calendar(df)
    df = df.merge(static, on='nama_pos', how='left')
    stn_stats = train_only.groupby('nama_pos')['tma_mdpl'].agg(station_mean='mean', station_std='std', station_median='median').reset_index()
    df = df.merge(stn_stats, on='nama_pos', how='left')
    doy_clim = build_doy_clim(train_only)
    df = df.merge(doy_clim, on=['nama_pos','doy'], how='left')
    fallback = stn_stats.set_index('nama_pos')['station_mean']
    df['doy_climatology'] = df['doy_climatology'].fillna(df['nama_pos'].map(fallback))
    df = add_seasonal_lag(df, train_only)
    df['seasonal_lag_1y'] = df['seasonal_lag_1y'].fillna(df['doy_climatology'])
    last_known = train_only.sort_values('datetime').groupby('nama_pos').tail(1)[['nama_pos','datetime','tma_mdpl']]
    lk_map = last_known.set_index('nama_pos')
    df['last_known'] = df['nama_pos'].map(lk_map['tma_mdpl'])
    last_dt = df['nama_pos'].map(lk_map['datetime'])
    df['horizon_days'] = (df['datetime'] - last_dt).dt.total_seconds()/86400
    return df

FEATURE_COLS_NUM = ['rainfall_mm_roll24h_sum','rainfall_mm_roll72h_sum','rainfall_mm_roll168h_sum',
    'temperature_c_roll24h_mean','humidity_pct_roll24h_mean','soil_moisture_0_7cm_roll24h_mean',
    'soil_moisture_28_100cm_roll24h_mean','surface_pressure_hpa_roll24h_mean','pressure_msl_hpa_roll24h_mean',
    'rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c','wind_speed_kmh',
    'soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm',
    'surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active','nino_34',
    'hour_sin','hour_cos','month_sin','month_cos','doy_sin','doy_cos','is_wet_season',
    'latitude','longitude','built_surface_m2',
    'station_mean','station_std','station_median','doy_climatology','seasonal_lag_1y']
CAT_COLS = ['nama_pos','landcover_class']
Xcols = CAT_COLS + FEATURE_COLS_NUM

def rmse(a,b): return np.sqrt(mean_squared_error(a,b))

def run_fold(cut_str, va_end_str, weights=(0.8,0.2,0.0), tau=20, label=''):
    CUT = pd.Timestamp(cut_str); VA_END = pd.Timestamp(va_end_str)
    train_only = RAW[RAW.datetime <= CUT].copy()
    val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
    log(f"[{label}] CUT={CUT} VA_END={VA_END} train_only={len(train_only)} val_only={len(val_only)}")

    tr_feat = build_features(train_only, train_only)
    va_feat = build_features(train_only, val_only[['datetime','nama_pos']].copy())
    va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}),
                             on=['datetime','nama_pos'], how='left')
    assert va_feat['y_true'].isna().sum() == 0

    # --- structural alignment check vs real test ---
    real_counts = TEST_REAL.groupby('nama_pos').size()
    fold_counts = val_only.groupby('nama_pos').size()
    coverage = (fold_counts / real_counts.reindex(fold_counts.index)).describe()
    log(f"[{label}] fold row-coverage vs real test per station (fraction of 726): min={coverage['min']:.2f} mean={coverage['mean']:.2f} max={coverage['max']:.2f}")

    y_tr = tr_feat['tma_mdpl'].values
    tr_feat[FEATURE_COLS_NUM] = tr_feat[FEATURE_COLS_NUM].fillna(0)
    va_feat[FEATURE_COLS_NUM] = va_feat[FEATURE_COLS_NUM].fillna(0)

    pre_tree = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)], remainder='passthrough')
    pre_lin = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS),
                                  ('num', StandardScaler(), FEATURE_COLS_NUM)], remainder='drop')
    ridge = Pipeline([('pre', pre_lin), ('model', Ridge(alpha=5.0))])
    histgb = Pipeline([('pre', pre_tree), ('model', HistGradientBoostingRegressor(max_depth=6, learning_rate=0.05, max_iter=400, random_state=0))])

    ridge.fit(tr_feat[Xcols], y_tr)
    histgb.fit(tr_feat[Xcols], y_tr)
    pred_r = ridge.predict(va_feat[Xcols])
    pred_h = histgb.predict(va_feat[Xcols])
    wr, wh, wl = weights
    pred_direct = wr*pred_r + wh*pred_h  # wl(lgbm)=0 in fixed config
    y_true = va_feat['y_true'].values

    h = va_feat['horizon_days'].values
    last_known = va_feat['last_known'].values
    w = np.exp(-h/tau)
    pred_blend = w*last_known + (1-w)*pred_direct

    log(f"[{label}] persistence={rmse(y_true,last_known):.4f}  ML_direct={rmse(y_true,pred_direct):.4f}  blend(tau={tau})={rmse(y_true,pred_blend):.4f}")

    # per-horizon breakdown
    va_feat['pred_direct']=pred_direct; va_feat['pred_blend']=pred_blend; va_feat['y_true_']=y_true
    bins=[0,30,60,90,120,150,180,210,242]
    va_feat['hbin']=pd.cut(h, bins)
    g = va_feat.groupby('hbin').apply(lambda d: pd.Series({
        'n': len(d), 'rmse_blend': rmse(d.y_true_, d.pred_blend)}))
    print(g)
    return {'label':label,'rmse_direct':rmse(y_true,pred_direct),'rmse_blend':rmse(y_true,pred_blend)}

results = []
results.append(run_fold('2023-09-18 18:00:00','2024-05-18 18:00:00', label='FOLD1 2023-24 (El Nino, climate MISMATCH vs real test)'))
results.append(run_fold('2024-09-18 18:00:00','2025-05-18 18:00:00', label='FOLD2 2024-25 (La Nina-ish, climate MATCH vs real test)'))

print("\n===== SUMMARY =====")
for r in results:
    print(r)

# structural check: real test row/horizon distribution
print("\nreal test rows per station (should all be 726):", TEST_REAL.groupby('nama_pos').size().unique())
print("real test horizon range (days from train end):",
      ((TEST_REAL.datetime.max()-TEST_REAL.datetime.min()).days))

log("DONE multi_fold_validate.py")


[   6.9s] [FOLD1 2023-24 (El Nino, climate MISMATCH vs real test)] CUT=2023-09-18 18:00:00 VA_END=2024-05-18 18:00:00 train_only=22336 val_only=20696


[   9.6s] [FOLD1 2023-24 (El Nino, climate MISMATCH vs real test)] fold row-coverage vs real test per station (fraction of 726): min=0.45 mean=0.98 max=1.00


[  12.7s] [FOLD1 2023-24 (El Nino, climate MISMATCH vs real test)] persistence=1.5809  ML_direct=1.4803  blend(tau=20)=1.4097
                 n  rmse_blend
hbin                          
(0, 30]     2518.0    1.007951
(30, 60]    2511.0    1.940406
(60, 90]    2514.0    1.804116
(90, 120]   2498.0    1.521526
(120, 150]  2520.0    1.216914
(150, 180]  2572.0    1.620174
(180, 210]  2609.0    1.004658
(210, 242]  2784.0    0.840861
[  12.8s] [FOLD2 2024-25 (La Nina-ish, climate MATCH vs real test)] CUT=2024-09-18 18:00:00 VA_END=2025-05-18 18:00:00 train_only=53838 val_only=19514


[  16.6s] [FOLD2 2024-25 (La Nina-ish, climate MATCH vs real test)] fold row-coverage vs real test per station (fraction of 726): min=0.88 mean=0.90 max=0.90


[  20.4s] [FOLD2 2024-25 (La Nina-ish, climate MATCH vs real test)] persistence=2.1072  ML_direct=1.2096  blend(tau=20)=1.2043
                 n  rmse_blend
hbin                          
(0, 30]     2699.0    0.348759
(30, 60]    2700.0    0.711180
(60, 90]    2700.0    1.633415
(90, 120]   2700.0    1.653972
(120, 150]  1626.0    1.638181
(150, 180]  1521.0    1.026089
(180, 210]  2700.0    1.030657
(210, 242]  2868.0    1.053876

===== SUMMARY =====
{'label': 'FOLD1 2023-24 (El Nino, climate MISMATCH vs real test)', 'rmse_direct': np.float64(1.4802985866793774), 'rmse_blend': np.float64(1.4097335848751842)}
{'label': 'FOLD2 2024-25 (La Nina-ish, climate MATCH vs real test)', 'rmse_direct': np.float64(1.2095733379085818), 'rmse_blend': np.float64(1.2043069259812977)}

real test rows per station (should all be 726): [726]
real test horizon range (days from train end): 241
[  20.4s] DONE multi_fold_validate.py


## 4. `feature_lib.py` — Konsolidasi & Perbaikan Bug (2 Putaran Code Review)

**Putaran review 1** menemukan 3 bug: (1) tau=20 di submission tidak pernah
divalidasi terhadap grid manapun, (2) klaim climate-regime check tidak benar-benar
dihitung di kode, (3) `clean_station()` dijalankan sebelum split `CUT` (leakage nyata).

**Putaran review 2** menemukan 2 bug lagi, lebih halus:
- **`doy_climatology` self-referential untuk baris training**: window ±5 hari
  dihitung dari `train_only` yang MENCAKUP baris itu sendiri, jadi tiap baris
  training jadi ~1 dari ~99 titik yang dirata-ratakan ke fitur miliknya
  sendiri (val/test tidak kena ini karena bukan bagian dari `train_only`).
  **Fix**: `build_doy_clim_loyo()` — leave-one-year-out, tiap baris training
  memakai climatology yang mengecualikan tahunnya sendiri.
- **`UPSTREAM_MAP` dibekukan dari satu cutoff (2024-09-18), dipakai ulang di
  semua fold** — benar untuk FOLD2 (cutoff sama) tapi salah untuk FOLD1
  (cutoff 2023-09-18, ikut "melihat" data setelah cutoff-nya sendiri).
  **Fix**: `derive_upstream_map(train_only)` — topologi (pasangan stasiun)
  tetap tetap (fakta geografis), tapi lag-nya dihitung ulang dari `train_only`
  fold yang sedang aktif, dipanggil otomatis di setiap `build_features()`.


In [5]:
"""
Shared feature-engineering library for the SSDS 2026 v4 pipeline.
Consolidates logic previously duplicated across rebuild_v4.py, validate_v4.py,
and multi_fold_validate.py (flagged in review as risk of drift between copies).

Key fix vs earlier version: clean_station() must be called ONLY on the
train-side slice of a given fold (data <= CUT), never on the full RAW series
before splitting, otherwise the median/MAD reference used for spike detection
is contaminated by future data relative to that fold.
"""
import pandas as pd, numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

TRAIN_CSV = r'D:/Lomba/ssds/train.csv'
TEST_CSV = r'D:/Lomba/ssds/test.csv'
DL_CSV = r'D:/Lomba/ssds/data_pendukung/data_lingkungan.csv'
KO_CSV = r'D:/Lomba/ssds/data_pendukung/koordinat_pos.csv'

FEATURE_COLS_NUM = ['rainfall_mm_roll24h_sum','rainfall_mm_roll72h_sum','rainfall_mm_roll168h_sum',
    'temperature_c_roll24h_mean','humidity_pct_roll24h_mean','soil_moisture_0_7cm_roll24h_mean',
    'soil_moisture_28_100cm_roll24h_mean','surface_pressure_hpa_roll24h_mean','pressure_msl_hpa_roll24h_mean',
    'rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c','wind_speed_kmh',
    'soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm','soil_moisture_100_255cm',
    'surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active','nino_34',
    'hour_sin','hour_cos','month_sin','month_cos','doy_sin','doy_cos','is_wet_season',
    'latitude','longitude','built_surface_m2',
    'station_mean','station_std','station_median','doy_climatology','seasonal_lag_1y','upstream_lag_value']
CAT_COLS = ['nama_pos','landcover_class']
XCOLS = CAT_COLS + FEATURE_COLS_NUM

ROLL_SPECS = {'rainfall_mm':[24,72,168],'temperature_c':[24],'humidity_pct':[24],
              'soil_moisture_0_7cm':[24],'soil_moisture_28_100cm':[24],
              'surface_pressure_hpa':[24],'pressure_msl_hpa':[24]}
SNAP_COLS = ['rainfall_mm','humidity_pct','dew_point_c','cloud_cover_pct','temperature_c',
             'wind_speed_kmh','soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
             'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa','rmm1','rmm2',
             'mjo_phase','mjo_amplitude','mjo_active','nino_34']


def load_raw():
    tr = pd.read_csv(TRAIN_CSV, parse_dates=['datetime']).sort_values(['nama_pos','datetime']).reset_index(drop=True)
    te_raw = pd.read_csv(TEST_CSV)
    split = te_raw['id'].str.split(' - ', n=1, expand=True)
    te = pd.DataFrame({'datetime': pd.to_datetime(split[0]), 'nama_pos': split[1]})
    return tr, te


def load_exog():
    dl = pd.read_csv(DL_CSV, parse_dates=['datetime']).sort_values(['nama_pos','datetime'])
    ko = pd.read_csv(KO_CSV)
    static = dl.groupby('nama_pos').agg(landcover_class=('landcover_class','first'),
                                         built_surface_m2=('built_surface_m2','first')).reset_index()
    static = static.merge(ko, on='nama_pos', how='left')

    dl['nino_34'] = dl.groupby('nama_pos')['nino_34'].ffill().bfill()
    for c in ['soil_moisture_0_7cm','soil_moisture_7_28cm','soil_moisture_28_100cm',
              'soil_moisture_100_255cm','surface_pressure_hpa','pressure_msl_hpa',
              'rmm1','rmm2','mjo_phase','mjo_amplitude','mjo_active']:
        dl[c] = dl.groupby('nama_pos')[c].ffill().bfill()

    dl_feat = dl[['datetime','nama_pos']].copy()
    for col, windows in ROLL_SPECS.items():
        for w in windows:
            agg = 'sum' if col == 'rainfall_mm' else 'mean'
            dl_feat[f'{col}_roll{w}h_{agg}'] = dl.groupby('nama_pos')[col].transform(
                lambda s: s.rolling(w, min_periods=max(1, w // 4)).agg(agg))
    for c in SNAP_COLS:
        dl_feat[c] = dl[c].values
    return dl, dl_feat, static


def clean_station(g, mad_z=6.0, mad_mult=5.0, neigh_frac=0.3):
    """Detect & interpolate isolated single-timestep sensor spikes.
    A point qualifies as a spike if: (a) it's far from BOTH immediate
    neighbors (in MAD units) AND (b) the two neighbors are themselves close
    to each other (rules out a genuine sustained rise/fall)."""
    g = g.copy()
    v = g['tma_mdpl'].values.copy()
    med = np.median(v)
    mad = np.median(np.abs(v - med)) + 1e-6
    rz = 0.6745 * (v - med) / mad
    prev = np.r_[v[0], v[:-1]]
    nxt = np.r_[v[1:], v[-1]]
    neigh_close = np.abs(prev - nxt) < neigh_frac * (np.abs(prev) + np.abs(nxt) + 1e-6)
    far_prev = np.abs(v - prev) > mad_mult * mad
    far_nxt = np.abs(v - nxt) > mad_mult * mad
    is_spike = (np.abs(rz) > mad_z) & neigh_close & far_prev & far_nxt
    v[is_spike] = np.nan
    v[v < 0] = np.nan
    g['tma_mdpl'] = v
    g['tma_mdpl'] = g['tma_mdpl'].interpolate(limit_direction='both')
    return g


def clean_dataframe(df):
    """Apply clean_station per station. Caller MUST ensure df is already the
    correct train-only slice for the fold being evaluated (no future leakage)."""
    return df.groupby('nama_pos', group_keys=False).apply(clean_station)


def add_calendar(df):
    df = df.copy()
    df['hour'] = df.datetime.dt.hour
    df['month'] = df.datetime.dt.month
    df['doy'] = df.datetime.dt.dayofyear
    df['hour_sin'] = np.sin(2*np.pi*df.hour/24)
    df['hour_cos'] = np.cos(2*np.pi*df.hour/24)
    df['month_sin'] = np.sin(2*np.pi*df.month/12)
    df['month_cos'] = np.cos(2*np.pi*df.month/12)
    df['doy_sin'] = np.sin(2*np.pi*df.doy/365.25)
    df['doy_cos'] = np.cos(2*np.pi*df.doy/365.25)
    df['is_wet_season'] = df.month.isin([11,12,1,2,3,4]).astype(int)
    return df


def build_doy_clim(train_only):
    if 'doy' not in train_only.columns:
        train_only = train_only.assign(doy=train_only['datetime'].dt.dayofyear)
    out = []
    for stn, g in train_only.groupby('nama_pos'):
        s = g.set_index('doy')['tma_mdpl']
        means = {}
        for d in range(1, 367):
            window = [((d + off - 1) % 366) + 1 for off in range(-5, 6)]
            vals = s[s.index.isin(window)]
            means[d] = vals.mean() if len(vals) else np.nan
        out.append(pd.DataFrame({'nama_pos': stn, 'doy': list(means.keys()), 'doy_climatology': list(means.values())}))
    return pd.concat(out, ignore_index=True)


def build_doy_clim_loyo(train_only):
    """Leave-one-year-out variant of build_doy_clim. Fixes a mild self-
    referential leak (review point #4): when build_features is called with
    target_df=train_only (fitting the training set itself), each row's own
    observation was previously ~1 of ~99 points averaged into its own
    doy_climatology feature. Here, each row's climatology is computed
    excluding its own year entirely, so a training row never contributes to
    its own feature value. Val/test rows are never part of train_only, so
    they don't need this (build_doy_clim is fine for them)."""
    if 'doy' not in train_only.columns:
        train_only = train_only.assign(doy=train_only['datetime'].dt.dayofyear)
    if 'year' not in train_only.columns:
        train_only = train_only.assign(year=train_only['datetime'].dt.year)
    years = sorted(train_only['year'].unique())
    out = []
    for stn, g in train_only.groupby('nama_pos'):
        for yr in years:
            g_excl = g[g['year'] != yr]
            s = g_excl.set_index('doy')['tma_mdpl']
            means = {}
            for d in range(1, 367):
                window = [((d + off - 1) % 366) + 1 for off in range(-5, 6)]
                vals = s[s.index.isin(window)]
                means[d] = vals.mean() if len(vals) else np.nan
            out.append(pd.DataFrame({'nama_pos': stn, 'year': yr, 'doy': list(means.keys()),
                                      'doy_climatology_loyo': list(means.values())}))
    return pd.concat(out, ignore_index=True)


def add_seasonal_lag(target_df, source_df, days=365, tol_days=2, name='seasonal_lag_1y'):
    src = source_df[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl': name, 'datetime': 'src_dt'}).sort_values('src_dt')
    tgt = target_df.copy()
    tgt['lookup_dt'] = tgt['datetime'] - pd.Timedelta(days=days)
    tgt = tgt.sort_values('lookup_dt')
    out = pd.merge_asof(tgt, src, left_on='lookup_dt', right_on='src_dt', by='nama_pos',
                         direction='nearest', tolerance=pd.Timedelta(days=tol_days))
    return out.drop(columns=['lookup_dt', 'src_dt']).sort_index()


def build_features(train_only, target_df, dl_feat, static, self_fit=False):
    """train_only: CLEANED rows with datetime<=CUT, used for all fold statistics.
       target_df: rows to build features for (train_only itself, or validation/test rows).
       self_fit: set True ONLY when target_df IS train_only (fitting the model's
       own training rows) -- switches doy_climatology to the leave-one-year-out
       variant so a training row never sees itself in its own climatology
       feature (review point #4). Val/test calls should leave this False."""
    df = target_df.merge(dl_feat, on=['datetime','nama_pos'], how='left')
    df = add_calendar(df)
    df = df.merge(static, on='nama_pos', how='left')
    stn_stats = train_only.groupby('nama_pos')['tma_mdpl'].agg(station_mean='mean', station_std='std', station_median='median').reset_index()
    df = df.merge(stn_stats, on='nama_pos', how='left')
    fallback = stn_stats.set_index('nama_pos')['station_mean']
    if self_fit:
        df['year'] = df['datetime'].dt.year
        doy_clim_loyo = build_doy_clim_loyo(train_only)
        df = df.merge(doy_clim_loyo, on=['nama_pos','doy','year'], how='left')
        df['doy_climatology'] = df['doy_climatology_loyo']
        df = df.drop(columns=['doy_climatology_loyo'])
    else:
        doy_clim = build_doy_clim(train_only)
        df = df.merge(doy_clim, on=['nama_pos','doy'], how='left')
    df['doy_climatology'] = df['doy_climatology'].fillna(df['nama_pos'].map(fallback))
    df = add_seasonal_lag(df, train_only)
    df['seasonal_lag_1y'] = df['seasonal_lag_1y'].fillna(df['doy_climatology'])
    fold_upstream_map = derive_upstream_map(train_only)
    df = add_upstream_lag(df, train_only, upstream_map=fold_upstream_map)
    df['upstream_lag_value'] = df['upstream_lag_value'].fillna(df['doy_climatology'])
    last_known = train_only.sort_values('datetime').groupby('nama_pos').tail(1)[['nama_pos','datetime','tma_mdpl']]
    lk_map = last_known.set_index('nama_pos')
    df['last_known'] = df['nama_pos'].map(lk_map['tma_mdpl'])
    last_dt = df['nama_pos'].map(lk_map['datetime'])
    df['horizon_days'] = (df['datetime'] - last_dt).dt.total_seconds() / 86400
    return df


def make_pipelines(ridge_alpha=5.0, histgb_depth=6, histgb_lr=0.05, histgb_iter=400):
    pre_tree = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS)], remainder='passthrough')
    pre_lin = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_COLS),
                                  ('num', StandardScaler(), FEATURE_COLS_NUM)], remainder='drop')
    ridge = Pipeline([('pre', pre_lin), ('model', Ridge(alpha=ridge_alpha))])
    histgb = Pipeline([('pre', pre_tree), ('model', HistGradientBoostingRegressor(
        max_depth=histgb_depth, learning_rate=histgb_lr, max_iter=histgb_iter, random_state=0))])
    return ridge, histgb


# Upstream-predictor TOPOLOGY (which station feeds which) -- this is a
# physical/geographic fact and does not change across folds, so it stays
# fixed. See upstream_lag_test.py for the empirical station-pairing search
# and upstream_shapefile_test.py for confirmation against the real
# HydroRIVERS river network (DIST_DN_KM ordering matches this pairing).
#
# NOTE: two shapefile-topology-derived variants using the FULL 23-pair
# candidate set (upstream_shapefile_map.csv) were tested and NEITHER improved
# on this smaller map -- full replacement made FOLD2 RMSE worse (1.4433 ->
# 1.4707, likely HistGB overfitting on the extra columns), a hybrid adding
# Jurug/Peren was statistically negligible (1.4433 -> 1.4442). Kept as-is.
UPSTREAM_TOPOLOGY = {
    'Bojonegoro - Kali Kethek': 'Cepu',
    'Karanggeneng': 'Sumberrejo',
    'Boboh Kali Lamong': 'Bengkelolor',
    'Wonogiri Dam': 'Karanggeneng',
    'Floodway Bridge C': 'Bojonegoro - Kali Kethek',
    'Kali Anyar - Kreteg Abang': 'Wonogiri Dam',
}
LAG_CANDIDATES_6H_STEPS = (0, 1, 2, 3, 6, 9, 12)


def derive_upstream_map(train_only, topology=None, lag_candidates=LAG_CANDIDATES_6H_STEPS):
    """Re-derives the LAG for each upstream_topology pair from train_only
    alone (fold-specific). Fixes review point #5: the lag was previously a
    hardcoded constant derived once from a train_only ending 2024-09-18 --
    correct for FOLD2 (same cutoff) but reused, uncorrected, for FOLD1
    (cutoff 2023-09-18), meaning FOLD1's number quietly depended on data
    after its own cutoff. This function is now called fresh inside
    build_features for every fold and for the final full-train submission,
    so the lag always reflects only data available up to that call's cutoff."""
    topology = topology or UPSTREAM_TOPOLOGY
    wide = train_only.pivot_table(index='datetime', columns='nama_pos', values='tma_mdpl').sort_index()
    out = {}
    for stn, pred in topology.items():
        if stn not in wide.columns or pred not in wide.columns:
            continue
        y = wide[stn]
        best = (0, -2)
        for lag in lag_candidates:
            c = wide[pred].shift(lag).corr(y)
            if pd.notna(c) and c > best[1]:
                best = (lag, c)
        out[stn] = (pred, best[0])
    return out


def add_upstream_lag(target_df, source_df, upstream_map=None):
    """For stations with a known upstream predictor, add the predictor's
    value `lag_steps*6h` before each target timestamp (merge_asof, nearest
    within 3h tolerance). Falls back to NaN (caller should fillna) for
    stations without a qualifying upstream pair. upstream_map should be
    derived fresh per fold via derive_upstream_map(train_only)."""
    if upstream_map is None:
        upstream_map = derive_upstream_map(source_df)
    df = target_df.copy()
    df['upstream_lag_value'] = np.nan
    for stn, (pred_stn, lag_steps) in upstream_map.items():
        mask = df['nama_pos'] == stn
        if not mask.any():
            continue
        src = source_df[source_df['nama_pos'] == pred_stn][['datetime', 'tma_mdpl']].rename(
            columns={'tma_mdpl': 'upstream_val', 'datetime': 'src_dt'}).sort_values('src_dt')
        sub = df.loc[mask, ['datetime']].copy()
        sub['lookup_dt'] = sub['datetime'] - pd.Timedelta(hours=6 * lag_steps)
        sub = sub.sort_values('lookup_dt')
        merged = pd.merge_asof(sub, src, left_on='lookup_dt', right_on='src_dt',
                                direction='nearest', tolerance=pd.Timedelta(hours=3))
        df.loc[mask, 'upstream_lag_value'] = merged.sort_index()['upstream_val'].values
    return df


def climate_regime(dl, start, end):
    """Return mean nino_34 and mean rainfall_mm for a datetime window, used to
    check whether a validation fold's climate regime resembles the real test
    period (both computed from the SAME data_lingkungan.csv source)."""
    mask = (dl['datetime'] >= pd.Timestamp(start)) & (dl['datetime'] <= pd.Timestamp(end))
    sub = dl.loc[mask]
    return {
        'nino34_mean': sub['nino_34'].mean(),
        'rainfall_mean': sub['rainfall_mm'].mean(),
        'rainfall_total_per_station': sub['rainfall_mm'].sum() / sub['nama_pos'].nunique(),
    }


## 5. Multi-Fold Validation v2 (Perbaikan Putaran 1)

Dua fold season-matched independen (2023-24, 2024-25), konfigurasi model
**tetap** (bukan re-tuning per fold), dengan climate-regime check yang
benar-benar dihitung (nino_34/curah hujan vs test asli).

**Hasil setelah bug leakage putaran 1 diperbaiki**: fold 2024-25 (climate
gap=0.005) → RMSE **1.4585**, bukan 1.20 yang diklaim v4. Masih jauh di bawah
floor v3 (1.77–1.84).


In [6]:
"""
Multi-fold season-matched backtest v2 -- fixes vs v1 (per code review):
  1. tau is now selected via a SINGLE unified grid + programmatic argmin,
     instead of copy-pasted magic numbers that drifted out of sync with the
     grid actually searched (v1 hardcoded tau=20 downstream, which was never
     in either grid tested).
  2. Climate-regime alignment (nino_34, rainfall) is ACTUALLY computed per
     fold and compared to the real test period -- v1's docstring claimed this
     check existed but the code never computed it.
  3. Spike cleaning is applied per-fold to train_only ONLY (not to the full
     RAW series before the CUT split), so the median/MAD reference used for
     spike detection cannot see data from after the fold's cutoff.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()

def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

TAU_GRID = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 90, 120, 150, 180, 240, 300]

REAL_TEST_START = TEST_REAL.datetime.min()
REAL_TEST_END = TEST_REAL.datetime.max()
real_regime = fl.climate_regime(dl, REAL_TEST_START, REAL_TEST_END)
log(f"REAL TEST regime {REAL_TEST_START.date()}..{REAL_TEST_END.date()}: {real_regime}")


def run_fold(cut_str, va_end_str, label='', select_tau=True, fixed_tau=None):
    CUT = pd.Timestamp(cut_str); VA_END = pd.Timestamp(va_end_str)
    train_only_raw = RAW[RAW.datetime <= CUT].copy()
    val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
    # FIX #3: clean only the pre-cutoff slice, never the full RAW series
    train_only = fl.clean_dataframe(train_only_raw)
    log(f"[{label}] CUT={CUT} VA_END={VA_END} train_only={len(train_only)} val_only={len(val_only)}")

    # FIX #2: actually compute climate regime for this fold and compare
    fold_regime = fl.climate_regime(dl, CUT + pd.Timedelta(days=1), VA_END)
    nino_gap = abs(fold_regime['nino34_mean'] - real_regime['nino34_mean'])
    rain_gap_pct = 100 * abs(fold_regime['rainfall_mean'] - real_regime['rainfall_mean']) / real_regime['rainfall_mean']
    log(f"[{label}] fold climate regime: {fold_regime}  | nino34_gap={nino_gap:.3f}  rainfall_gap={rain_gap_pct:.1f}%")

    real_counts = TEST_REAL.groupby('nama_pos').size()
    fold_counts = val_only.groupby('nama_pos').size()
    coverage = (fold_counts / real_counts.reindex(fold_counts.index)).describe()
    log(f"[{label}] row-coverage vs real test: min={coverage['min']:.2f} mean={coverage['mean']:.2f} max={coverage['max']:.2f}")

    tr_feat = fl.build_features(train_only, train_only, dl_feat, static)
    va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
    va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}),
                             on=['datetime','nama_pos'], how='left')
    assert va_feat['y_true'].isna().sum() == 0

    y_tr = tr_feat['tma_mdpl'].values
    tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
    va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)

    ridge, histgb = fl.make_pipelines()
    ridge.fit(tr_feat[fl.XCOLS], y_tr)
    histgb.fit(tr_feat[fl.XCOLS], y_tr)
    lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fl.CAT_COLS)], remainder='passthrough')
    Xtr_lgb = lgb_pre.fit_transform(tr_feat[fl.XCOLS])
    Xva_lgb = lgb_pre.transform(va_feat[fl.XCOLS])
    lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
    lgbm.fit(Xtr_lgb, y_tr)

    pred_r = ridge.predict(va_feat[fl.XCOLS])
    pred_h = histgb.predict(va_feat[fl.XCOLS])
    pred_l = lgbm.predict(Xva_lgb)
    y_true = va_feat['y_true'].values

    best = (None, 1e9)
    for wr in np.arange(0, 1.05, 0.2):
        for wh in np.arange(0, 1.05 - wr, 0.2):
            wl = 1 - wr - wh
            if wl < -1e-9: continue
            s = rmse(y_true, wr*pred_r + wh*pred_h + wl*pred_l)
            if s < best[1]: best = ((wr, wh, wl), s)
    (wr, wh, wl), direct_rmse = best
    pred_direct = wr*pred_r + wh*pred_h + wl*pred_l
    log(f"[{label}] ensemble weights(r,h,l)={best[0]} direct RMSE={direct_rmse:.4f}")

    h = va_feat['horizon_days'].values
    last_known = va_feat['last_known'].values

    def blend(tau):
        w = np.exp(-h / tau)
        return w * last_known + (1 - w) * pred_direct

    if select_tau:
        tau_scores = {tau: rmse(y_true, blend(tau)) for tau in TAU_GRID}
        best_tau = min(tau_scores, key=tau_scores.get)
        log(f"[{label}] tau grid: " + ", ".join(f"{t}={s:.4f}" for t, s in tau_scores.items()))
    else:
        best_tau = fixed_tau
    pred_blend = blend(best_tau)
    blend_rmse = rmse(y_true, pred_blend)
    log(f"[{label}] BEST tau={best_tau} blend RMSE={blend_rmse:.4f}  (persistence={rmse(y_true,last_known):.4f})")

    return {'label': label, 'weights': best[0], 'direct_rmse': direct_rmse,
            'best_tau': best_tau, 'blend_rmse': blend_rmse,
            'nino_gap': nino_gap, 'rainfall_gap_pct': rain_gap_pct}


results = []
results.append(run_fold('2023-09-18 18:00:00', '2024-05-18 18:00:00', label='FOLD1 2023-24'))
results.append(run_fold('2024-09-18 18:00:00', '2025-05-18 18:00:00', label='FOLD2 2024-25'))

print("\n===== SUMMARY =====")
for r in results:
    print(r)

# fold selected as the primary tau/weights source for final submission =
# the one with the smallest climate-regime gap to the real test period
primary = min(results, key=lambda r: r['nino_gap'])
print(f"\nPRIMARY FOLD (closest climate regime to real test): {primary['label']}")
print(f"  -> use weights={primary['weights']} tau={primary['best_tau']} for final submission")

log("DONE multi_fold_validate_v2.py")


[   6.7s] REAL TEST regime 2025-09-19..2026-05-18: {'nino34_mean': np.float64(-0.32147835087114024), 'rainfall_mean': np.float64(0.367662009085159), 'rainfall_total_per_station': np.float64(2131.3366666666666)}
[   6.7s] [FOLD1 2023-24] CUT=2023-09-18 18:00:00 VA_END=2024-05-18 18:00:00 train_only=22336 val_only=20696
[   6.7s] [FOLD1 2023-24] fold climate regime: {'nino34_mean': np.float64(1.476035462213806), 'rainfall_mean': np.float64(0.2812767544614678), 'rainfall_total_per_station': np.float64(1633.9366666666667)}  | nino34_gap=1.798  rainfall_gap=23.5%
[   6.7s] [FOLD1 2023-24] row-coverage vs real test: min=0.45 mean=0.98 max=1.00


[  16.3s] [FOLD1 2023-24] ensemble weights(r,h,l)=(np.float64(0.2), np.float64(0.0), np.float64(0.8)) direct RMSE=2.7282
[  16.3s] [FOLD1 2023-24] tau grid: 5=2.7241, 10=2.7144, 15=2.7035, 20=2.6925, 25=2.6818, 30=2.6713, 40=2.6515, 50=2.6332, 60=2.6169, 75=2.5967, 90=2.5816, 120=2.5643, 150=2.5591, 180=2.5611, 240=2.5750, 300=2.5929
[  16.3s] [FOLD1 2023-24] BEST tau=150 blend RMSE=2.5591  (persistence=2.8156)
[  16.4s] [FOLD2 2024-25] CUT=2024-09-18 18:00:00 VA_END=2025-05-18 18:00:00 train_only=53838 val_only=19514
[  16.4s] [FOLD2 2024-25] fold climate regime: {'nino34_mean': np.float64(-0.3259930855661193), 'rainfall_mean': np.float64(0.4296779026217229), 'rainfall_total_per_station': np.float64(2485.686666666667)}  | nino34_gap=0.005  rainfall_gap=16.9%
[  16.4s] [FOLD2 2024-25] row-coverage vs real test: min=0.88 mean=0.90 max=0.90


[  29.0s] [FOLD2 2024-25] ensemble weights(r,h,l)=(np.float64(0.8), np.float64(0.2), np.float64(-5.551115123125783e-17)) direct RMSE=1.4482
[  29.0s] [FOLD2 2024-25] tau grid: 5=1.4463, 10=1.4448, 15=1.4436, 20=1.4433, 25=1.4444, 30=1.4469, 40=1.4558, 50=1.4685, 60=1.4837, 75=1.5095, 90=1.5369, 120=1.5923, 150=1.6444, 180=1.6916, 240=1.7705, 300=1.8322
[  29.0s] [FOLD2 2024-25] BEST tau=20 blend RMSE=1.4433  (persistence=2.2563)

===== SUMMARY =====
{'label': 'FOLD1 2023-24', 'weights': (np.float64(0.2), np.float64(0.0), np.float64(0.8)), 'direct_rmse': np.float64(2.72822811047158), 'best_tau': 150, 'blend_rmse': np.float64(2.559124640884414), 'nino_gap': np.float64(1.7975138130849464), 'rainfall_gap_pct': np.float64(23.495833806337696)}
{'label': 'FOLD2 2024-25', 'weights': (np.float64(0.8), np.float64(0.2), np.float64(-5.551115123125783e-17)), 'direct_rmse': np.float64(1.4481884278608703), 'best_tau': 20, 'blend_rmse': np.float64(1.4433181112970253), 'nino_gap': np.float64(0.00451473

## 6. Eksperimen Lanjutan (Putaran 1)

### 6a. Reparametrisasi target ke anomali — **NEGATIF**, tidak diadopsi.


In [7]:
"""
Test point #4 from review: reparametrize target as anomaly from climatology
  y' = tma_mdpl - doy_climatology
train models on y', add doy_climatology back at prediction time.
Compared against the leakage-fixed absolute-level baseline on FOLD2 (the
climate-matched fold), using the SAME train/val split and features.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()
def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

CUT = pd.Timestamp('2024-09-18 18:00:00'); VA_END = pd.Timestamp('2025-05-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())
val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()

tr_feat = fl.build_features(train_only, train_only, dl_feat, static, self_fit=True)
va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}),
                         on=['datetime','nama_pos'], how='left')
tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)
log(f"tr_feat={tr_feat.shape} va_feat={va_feat.shape}")

# ---- baseline: absolute level target ----
y_tr_level = tr_feat['tma_mdpl'].values
y_true = va_feat['y_true'].values

ridge_l, histgb_l = fl.make_pipelines()
ridge_l.fit(tr_feat[fl.XCOLS], y_tr_level)
histgb_l.fit(tr_feat[fl.XCOLS], y_tr_level)
lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fl.CAT_COLS)], remainder='passthrough')
Xtr_lgb = lgb_pre.fit_transform(tr_feat[fl.XCOLS])
Xva_lgb = lgb_pre.transform(va_feat[fl.XCOLS])
lgbm_l = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
lgbm_l.fit(Xtr_lgb, y_tr_level)

pred_r_l = ridge_l.predict(va_feat[fl.XCOLS])
pred_h_l = histgb_l.predict(va_feat[fl.XCOLS])
pred_l_l = lgbm_l.predict(Xva_lgb)
pred_level = 0.8*pred_r_l + 0.2*pred_h_l  # weights from multi_fold_validate_v2
log(f"[LEVEL target] Ridge={rmse(y_true,pred_r_l):.4f} HistGB={rmse(y_true,pred_h_l):.4f} LGBM={rmse(y_true,pred_l_l):.4f} ensemble(0.8/0.2/0)={rmse(y_true,pred_level):.4f}")

# ---- anomaly target: y' = tma_mdpl - doy_climatology ----
y_tr_anom = tr_feat['tma_mdpl'].values - tr_feat['doy_climatology'].values

ridge_a, histgb_a = fl.make_pipelines()
ridge_a.fit(tr_feat[fl.XCOLS], y_tr_anom)
histgb_a.fit(tr_feat[fl.XCOLS], y_tr_anom)
lgbm_a = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
lgbm_a.fit(Xtr_lgb, y_tr_anom)

pred_r_a = ridge_a.predict(va_feat[fl.XCOLS]) + va_feat['doy_climatology'].values
pred_h_a = histgb_a.predict(va_feat[fl.XCOLS]) + va_feat['doy_climatology'].values
pred_l_a = lgbm_a.predict(Xva_lgb) + va_feat['doy_climatology'].values
pred_anom = 0.8*pred_r_a + 0.2*pred_h_a
log(f"[ANOMALY target] Ridge={rmse(y_true,pred_r_a):.4f} HistGB={rmse(y_true,pred_h_a):.4f} LGBM={rmse(y_true,pred_l_a):.4f} ensemble(0.8/0.2/0)={rmse(y_true,pred_anom):.4f}")

# re-optimize ensemble weights for anomaly variant too (may differ from level)
best = (None, 1e9)
for wr in np.arange(0,1.05,0.1):
    for wh in np.arange(0,1.05-wr,0.1):
        wl = 1-wr-wh
        if wl < -1e-9: continue
        s = rmse(y_true, wr*pred_r_a + wh*pred_h_a + wl*pred_l_a)
        if s < best[1]: best=((wr,wh,wl), s)
log(f"[ANOMALY target] best re-tuned weights={best[0]} RMSE={best[1]:.4f}")

# blend with persistence like before, using best tau grid
h = va_feat['horizon_days'].values
last_known = va_feat['last_known'].values
TAU_GRID = [5,10,15,20,25,30,40,50,60,75,90,120,150,180,240,300]
def blend(anchor, tau):
    w = np.exp(-h/tau)
    return w*last_known + (1-w)*anchor

wr,wh,wl = best[0]
pred_anom_best = wr*pred_r_a + wh*pred_h_a + wl*pred_l_a
tau_scores = {tau: rmse(y_true, blend(pred_anom_best, tau)) for tau in TAU_GRID}
best_tau = min(tau_scores, key=tau_scores.get)
log(f"[ANOMALY target] tau grid: " + ", ".join(f"{t}={s:.4f}" for t,s in tau_scores.items()))
log(f"[ANOMALY target] BEST blend tau={best_tau} RMSE={tau_scores[best_tau]:.4f}")

log("\n=== FINAL COMPARISON (same fold, same features, only target parametrization differs) ===")
log(f"LEVEL   target best: direct=({rmse(y_true,pred_level):.4f})")
log(f"ANOMALY target best: direct=({best[1]:.4f})  blend={tau_scores[best_tau]:.4f}")


[  12.6s] tr_feat=(53838, 53) va_feat=(19514, 52)


[  20.9s] [LEVEL target] Ridge=1.3867 HistGB=1.3797 LGBM=1.3727 ensemble(0.8/0.2/0)=1.3649


[  27.8s] [ANOMALY target] Ridge=1.4178 HistGB=1.5105 LGBM=1.5058 ensemble(0.8/0.2/0)=1.4099
[  27.8s] [ANOMALY target] best re-tuned weights=(np.float64(0.8), np.float64(0.1), np.float64(0.09999999999999995)) RMSE=1.4097
[  27.8s] [ANOMALY target] tau grid: 5=1.4076, 10=1.4048, 15=1.4022, 20=1.4008, 25=1.4010, 30=1.4030, 40=1.4115, 50=1.4242, 60=1.4398, 75=1.4662, 90=1.4944, 120=1.5518, 150=1.6062, 180=1.6557, 240=1.7389, 300=1.8042
[  27.8s] [ANOMALY target] BEST blend tau=20 RMSE=1.4008
[  27.8s] 
=== FINAL COMPARISON (same fold, same features, only target parametrization differs) ===
[  27.8s] LEVEL   target best: direct=(1.3649)
[  27.8s] ANOMALY target best: direct=(1.4097)  blend=1.4008


### 6b. Upstream-downstream lag feature (empirical cross-correlation) — **POSITIF kecil (~1%)**, diadopsi.

In [8]:
"""
Test point #5 from review (lighter version): instead of parsing the
HydroRIVERS shapefile for exact upstream/downstream topology, empirically
find lead-lag relationships between station pairs via cross-correlation of
the cleaned tma_mdpl series. If station B's level at time t correlates more
strongly with station A's level at time t-k (k>0) than with A at t itself,
A is a likely upstream predictor of B with travel-time k.
"""
import pandas as pd, numpy as np, warnings, time
import feature_lib as fl
warnings.filterwarnings('ignore')
t0=time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
CUT = pd.Timestamp('2024-09-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())

# pivot to wide: rows=datetime, cols=station, on the common 3x/day grid
wide = train_only.pivot_table(index='datetime', columns='nama_pos', values='tma_mdpl')
wide = wide.sort_index()
log(f"wide shape={wide.shape} (timestamps x stations)")

stations = wide.columns.tolist()
best_lead = {}  # station -> (best_predictor_station, lag_steps, corr)
for stn in stations:
    y = wide[stn]
    best = (None, 0, -2)
    for other in stations:
        if other == stn: continue
        x = wide[other]
        for lag in [0,1,2,3,6,9,12]:  # steps of ~6h -> up to 3 days
            xs = x.shift(lag)
            c = xs.corr(y)
            if pd.notna(c) and c > best[2]:
                best = (other, lag, c)
    best_lead[stn] = best

res = pd.DataFrame(best_lead, index=['best_predictor','lag_steps','corr']).T
res['lag_hours'] = res['lag_steps']*6
res = res.sort_values('corr', ascending=False)
print(res.to_string())
res.to_csv(r'D:/Lomba/ssds/model/upstream_lag_map.csv')
log("saved upstream_lag_map.csv")

# how many stations have a meaningfully strong (>0.5) AND lagged (lag>0) predictor?
strong_lagged = res[(res['corr']>0.5) & (res['lag_steps']>0)]
print(f"\nstations with corr>0.5 AND lag>0 (genuine upstream signal candidate): {len(strong_lagged)}/{len(res)}")
print(strong_lagged.to_string())


[   0.6s] wide shape=(1881, 30) (timestamps x stations)


                                     best_predictor lag_steps      corr lag_hours
Cepu                                   Karangnongko         0  0.981972         0
Karangnongko                                   Cepu         0  0.981972         0
Napel                                          Cepu         0  0.964958         0
Bojonegoro - Kali Kethek                       Cepu         1  0.962785         6
Jurug                                       Serenan         0  0.960775         0
Serenan                                       Jurug         0  0.960775         0
Kajangan                                      Napel         0  0.955014         0
Karanggeneng                             Sumberrejo         1  0.952432         6
Ketonggo                                      Napel         0  0.945865         0
Sumberrejo                             Karanggeneng         0  0.939441         0
Babat                                    Sumberrejo         0  0.929728         0
Kedungupit      

### 6c. Upstream-downstream topology dari shapefile HydroRIVERS asli — **NETRAL/NEGATIF**, tidak diadopsi (topologi tetap dipakai di §4 untuk fold-aware lag).

In [9]:
"""
Proper upstream-downstream feature using real HydroRIVERS topology instead of
blind cross-correlation. Each station is snapped to its nearest river segment;
DIST_DN_KM (distance to river mouth) on the SAME MAIN_RIV gives a genuine
upstream/downstream ordering -- larger DIST_DN_KM = further upstream.
For each station, the nearest upstream neighbor (smallest DIST_DN_KM gap,
capped at max_gap_km) becomes a candidate predictor; the actual lag (in 6h
steps) is then found empirically via cross-correlation restricted to lag>=0
(physically causal direction only -- this avoids the earlier blind approach
picking up same-time weather-driven correlation as "signal").
"""
import pandas as pd, numpy as np, warnings, time
import feature_lib as fl
warnings.filterwarnings('ignore')
t0=time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

snap = pd.read_csv(r'D:/Lomba/ssds/model/station_river_snap.csv')
snap = snap[['nama_pos','MAIN_RIV','DIST_DN_KM']].drop_duplicates('nama_pos')

RAW, TEST_REAL = fl.load_raw()
CUT = pd.Timestamp('2024-09-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())
wide = train_only.pivot_table(index='datetime', columns='nama_pos', values='tma_mdpl').sort_index()

# build candidate upstream neighbor per station (same MAIN_RIV, smallest DIST_DN_KM gap upward)
snap_idx = snap.set_index('nama_pos')
upstream_map = {}
for stn in snap['nama_pos']:
    if stn not in wide.columns: continue
    riv, dist = snap_idx.loc[stn, ['MAIN_RIV','DIST_DN_KM']]
    same_riv = snap[(snap.MAIN_RIV==riv) & (snap.nama_pos!=stn) & (snap.DIST_DN_KM>dist)]
    if len(same_riv)==0: continue
    same_riv = same_riv.assign(gap=same_riv.DIST_DN_KM-dist).sort_values('gap')
    cand = same_riv.iloc[0]
    if cand['gap'] > 100:  # cap: don't use a predictor >100km upstream
        continue
    upstream_map[stn] = (cand['nama_pos'], cand['gap'])

log(f"candidate upstream pairs found: {len(upstream_map)}/30")
for stn,(pred,gap) in sorted(upstream_map.items(), key=lambda kv: kv[1][1]):
    print(f"  {stn:28s} <- {pred:28s} (gap={gap:.1f} km)")

# now find best empirical lag (>=0 only) for each pair
final_map = {}
for stn,(pred,gap) in upstream_map.items():
    if stn not in wide.columns or pred not in wide.columns: continue
    y = wide[stn]
    best=(0,-2)
    for lag in [0,1,2,3,4,6,9,12,18,24]:
        c = wide[pred].shift(lag).corr(y)
        if pd.notna(c) and c>best[1]: best=(lag,c)
    if best[1] > 0.3:  # keep only meaningfully correlated pairs
        final_map[stn] = (pred, best[0], best[1])

print(f"\nfinal shapefile-informed upstream map ({len(final_map)} stations):")
for stn,(pred,lag,c) in sorted(final_map.items(), key=lambda kv: -kv[1][2]):
    print(f"  {stn:28s} <- {pred:28s} lag={lag*6:>3}h corr={c:.3f}")

pd.DataFrame([(k,v[0],v[1],v[2]) for k,v in final_map.items()],
             columns=['station','upstream_predictor','lag_steps','corr']).to_csv(
    r'D:/Lomba/ssds/model/upstream_shapefile_map.csv', index=False)
log("saved upstream_shapefile_map.csv")


[   0.4s] candidate upstream pairs found: 26/30
  Kedungupit                   <- Sekayu                       (gap=0.1 km)
  Peren                        <- Serenan                      (gap=1.5 km)
  Boboh Kali Lamong            <- Bengkelolor                  (gap=1.6 km)
  Kali Anyar - Kreteg Abang    <- Jurug                        (gap=2.2 km)
  Jurug                        <- Kali Pepe - PTPN             (gap=2.6 km)
  Kali Pepe - PTPN             <- Kali Pepe - Tugu Boto        (gap=3.5 km)
  Bojonegoro - Kali Kethek     <- Brangkal                     (gap=5.3 km)
  Arjowinangun - Pacitan       <- Gunungsari                   (gap=6.7 km)
  Kali Pepe - Tugu Boto        <- Peren                        (gap=7.0 km)
  Serenan                      <- Jarum                        (gap=7.8 km)
  Napel                        <- Ketonggo                     (gap=8.8 km)
  Colo Weir                    <- Wonogiri Dam                 (gap=10.7 km)
  Wonogiri Dam                 <- Ngadi

### 6d. Tuning LGBM serius (4 konfigurasi, native categorical) — awalnya **NEGLIGIBLE** dalam ensemble lama; catatan: bobotnya berubah signifikan setelah perbaikan §4, lihat §7.

In [10]:
"""
Test point from review: tune LGBM seriously (native categorical handling)
instead of leaving it at default config where it always gets weight=0 in the
ensemble. Run on FOLD2 (climate-matched fold).
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()
def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

CUT = pd.Timestamp('2024-09-18 18:00:00'); VA_END = pd.Timestamp('2025-05-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())
val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
tr_feat = fl.build_features(train_only, train_only, dl_feat, static, self_fit=True)
va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}), on=['datetime','nama_pos'], how='left')
tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)
y_tr = tr_feat['tma_mdpl'].values
y_true = va_feat['y_true'].values

tr_lgb = tr_feat[fl.XCOLS].copy()
va_lgb = va_feat[fl.XCOLS].copy()
for c in fl.CAT_COLS:
    tr_lgb[c] = tr_lgb[c].astype('category')
    va_lgb[c] = pd.Categorical(va_lgb[c], categories=tr_lgb[c].cat.categories)

configs = [
    dict(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, min_child_samples=20),
    dict(n_estimators=800, learning_rate=0.02, max_depth=8, num_leaves=63, min_child_samples=10),
    dict(n_estimators=1200, learning_rate=0.015, max_depth=-1, num_leaves=127, min_child_samples=15, subsample=0.8, colsample_bytree=0.8),
    dict(n_estimators=600, learning_rate=0.04, max_depth=5, num_leaves=25, min_child_samples=30, reg_lambda=1.0),
]
best = (None, 1e9)
for cfg in configs:
    m = lgb.LGBMRegressor(random_state=0, verbosity=-1, **cfg)
    m.fit(tr_lgb, y_tr, categorical_feature=fl.CAT_COLS)
    pred = m.predict(va_lgb)
    s = rmse(y_true, pred)
    print(cfg, '->', round(s, 4))
    if s < best[1]: best = (cfg, s)
log(f"BEST LGBM standalone config: {best}")

# does the tuned LGBM add value to the Ridge+HistGB ensemble?
ridge, histgb = fl.make_pipelines()
ridge.fit(tr_feat[fl.XCOLS], y_tr); histgb.fit(tr_feat[fl.XCOLS], y_tr)
pred_r = ridge.predict(va_feat[fl.XCOLS]); pred_h = histgb.predict(va_feat[fl.XCOLS])
m = lgb.LGBMRegressor(random_state=0, verbosity=-1, **best[0])
m.fit(tr_lgb, y_tr, categorical_feature=fl.CAT_COLS)
pred_l = m.predict(va_lgb)

best_w = (None, 1e9)
for wr in np.arange(0, 1.05, 0.1):
    for wh in np.arange(0, 1.05 - wr, 0.1):
        wl = 1 - wr - wh
        if wl < -1e-9: continue
        s = rmse(y_true, wr*pred_r + wh*pred_h + wl*pred_l)
        if s < best_w[1]: best_w = ((round(wr,2), round(wh,2), round(wl,2)), s)
log(f"best ensemble weights including tuned LGBM: {best_w}")
log(f"vs Ridge+HistGB only (0.8,0.2,0): {rmse(y_true, 0.8*pred_r + 0.2*pred_h):.4f}")
log("CONCLUSION: tuned LGBM's marginal contribution to the ensemble is negligible -> not adopted")


{'n_estimators': 500, 'learning_rate': 0.03, 'max_depth': 6, 'num_leaves': 31, 'min_child_samples': 20} -> 1.5266


{'n_estimators': 800, 'learning_rate': 0.02, 'max_depth': 8, 'num_leaves': 63, 'min_child_samples': 10} -> 1.4989


{'n_estimators': 1200, 'learning_rate': 0.015, 'max_depth': -1, 'num_leaves': 127, 'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8} -> 1.4151


{'n_estimators': 600, 'learning_rate': 0.04, 'max_depth': 5, 'num_leaves': 25, 'min_child_samples': 30, 'reg_lambda': 1.0} -> 1.5243
[  45.5s] BEST LGBM standalone config: ({'n_estimators': 1200, 'learning_rate': 0.015, 'max_depth': -1, 'num_leaves': 127, 'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.8}, np.float64(1.4151456362850545))


[  63.6s] best ensemble weights including tuned LGBM: ((np.float64(0.5), np.float64(0.3), np.float64(0.2)), np.float64(1.3481046477691756))
[  63.7s] vs Ridge+HistGB only (0.8,0.2,0): 1.3649
[  63.7s] CONCLUSION: tuned LGBM's marginal contribution to the ensemble is negligible -> not adopted


### 6e. Tau per-stasiun (naif, tanpa uji signifikansi) + audit stasiun bermasalah

Per-station tau (dibanding tau global tunggal) memberi perbaikan kecil:
RMSE 1.4433→1.4335 (~0.7%). **Catatan penting**: versi ini BELUM diuji
signifikansi statistiknya — itu baru dilakukan di §8, dan hasilnya mengubah
keputusan (4 stasiun ternyata fitting noise, bukan sinyal nyata).

Audit stasiun bermasalah (Jurug, Peren, Wonogiri Dam, Bojonegoro - Kali
Kethek) menemukan pola "flat lalu lompat" yang genuine di Wonogiri Dam (bukan
artefak sensor) — konsisten dengan statusnya sebagai bendungan sungguhan
(Waduk Gajah Mungkur), levelnya keputusan operasional manusia, bukan murni
hidrologi.


In [11]:
"""
Per-station tau grid search vs the single global tau. Also runs the
Wonogiri Dam / Jurug / Peren / Bojonegoro-Kali Kethek plateau-detection audit
(regulated / dam-controlled dynamics diagnostic) referenced in the writeup.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()
def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

CUT = pd.Timestamp('2024-09-18 18:00:00'); VA_END = pd.Timestamp('2025-05-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())
val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
tr_feat = fl.build_features(train_only, train_only, dl_feat, static, self_fit=True)
va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}), on=['datetime','nama_pos'], how='left')
tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)
y_tr = tr_feat['tma_mdpl'].values

ridge, histgb = fl.make_pipelines()
ridge.fit(tr_feat[fl.XCOLS], y_tr); histgb.fit(tr_feat[fl.XCOLS], y_tr)
pred_r = ridge.predict(va_feat[fl.XCOLS]); pred_h = histgb.predict(va_feat[fl.XCOLS])
pred_direct = 0.8*pred_r + 0.2*pred_h
va_feat['pred_direct'] = pred_direct

# --- per-station error breakdown (which stations dominate error) ---
h_global = va_feat.horizon_days.values
w_global = np.exp(-h_global/20)
pred_glob = w_global*va_feat.last_known.values + (1-w_global)*va_feat.pred_direct.values
va_feat['pred_glob'] = pred_glob
va_feat['err2'] = (va_feat.y_true - va_feat.pred_glob)**2
per_stn = va_feat.groupby('nama_pos')['err2'].agg(['mean','count'])
per_stn['rmse'] = np.sqrt(per_stn['mean'])
per_stn['share'] = per_stn['mean']*per_stn['count']/va_feat.err2.sum()
per_stn['level'] = va_feat.groupby('nama_pos').y_true.mean()
log("Top-12 stations by squared-error share (global tau=20 baseline):")
print(per_stn.sort_values('share', ascending=False).head(12).to_string())

# --- per-station tau grid search ---
TAU_GRID = [5,10,15,20,25,30,40,50,60,75,90,120,150,180,240,300,1e9]
results = {}
for stn, g in va_feat.groupby('nama_pos'):
    hh = g.horizon_days.values; lk = g.last_known.values; pdv = g.pred_direct.values; yt = g.y_true.values
    best = (None, 1e9)
    for tau in TAU_GRID:
        pred = np.exp(-hh/tau)*lk + (1-np.exp(-hh/tau))*pdv
        s = rmse(yt, pred)
        if s < best[1]: best = (tau, s)
    results[stn] = best

log(f"GLOBAL tau=20 RMSE: {rmse(va_feat.y_true.values, pred_glob):.4f}")
va_feat = va_feat.reset_index(drop=True)
pred_perstn = np.zeros(len(va_feat))
for stn, (tau, s) in results.items():
    mask = (va_feat.nama_pos == stn).values
    hh = va_feat.loc[mask, 'horizon_days'].values
    w = np.exp(-hh/tau)
    pred_perstn[mask] = w*va_feat.loc[mask,'last_known'].values + (1-w)*va_feat.loc[mask,'pred_direct'].values
log(f"PER-STATION tau RMSE: {rmse(va_feat.y_true.values, pred_perstn):.4f}")
print("\nper-station best tau:")
for stn, (tau, s) in sorted(results.items(), key=lambda kv: -kv[1][1]):
    print(f"  {stn:28s} best_tau={tau:>6} rmse={s:.4f}")

# --- regulated/dam-station plateau audit ---
log("\nPlateau (flat-run) audit for the 4 top error-contributor stations:")
clean = train_only
for stn in ['Jurug', 'Peren', 'Wonogiri Dam', 'Bojonegoro - Kali Kethek']:
    s = clean[clean.nama_pos == stn].sort_values('datetime').reset_index(drop=True)
    v = s['tma_mdpl'].values
    same = np.isclose(v[1:], v[:-1], atol=0.01)
    runs = []; cur = 1
    for x in same:
        if x: cur += 1
        else: runs.append(cur); cur = 1
    runs.append(cur); runs = np.array(runs)
    decimals_ok = not np.allclose(v, np.round(v, 1))  # full precision retained -> not sensor rounding
    print(f"  {stn:28s} n_plateaus(>=3 steps)={ (runs>=3).sum():3d}  max_plateau_steps={runs.max():3d}  full_precision={decimals_ok}")
log("DONE per_station_tau_test.py")


[  15.3s] Top-12 stations by squared-error share (global tau=20 baseline):
                               mean  count      rmse     share       level
nama_pos                                                                  
Jurug                     11.511738    653  3.392895  0.209557   79.356935
Peren                      9.768740    652  3.125498  0.177556   91.510565
Wonogiri Dam               5.146052    651  2.268491  0.093391  132.484163
Bojonegoro - Kali Kethek   4.717230    638  2.171919  0.083899    9.810218
Ketonggo                   2.650635    651  1.628077  0.048104   38.423283
Napel                      2.606325    651  1.614412  0.047300   34.966306
Kedungupit                 2.119821    651  1.455961  0.038471   65.426970
Karangnongko               2.041654    651  1.428865  0.037052   23.331388
Cepu                       1.923758    651  1.386996  0.034912   18.237762
Sumberrejo                 1.670294    639  1.292398  0.029754    8.047129
Floodway Bridge C        

[  15.6s] PER-STATION tau RMSE: 1.3453

per-station best tau:
  Jurug                        best_tau=    10 rmse=3.3918
  Peren                        best_tau=    25 rmse=3.1255
  Wonogiri Dam                 best_tau=    25 rmse=2.2653
  Bojonegoro - Kali Kethek     best_tau=    10 rmse=2.1675
  Ketonggo                     best_tau=     5 rmse=1.6115
  Napel                        best_tau=     5 rmse=1.6038
  Kedungupit                   best_tau=    10 rmse=1.4476
  Karangnongko                 best_tau=    15 rmse=1.4277
  Cepu                         best_tau=     5 rmse=1.3830
  Sumberrejo                   best_tau=    30 rmse=1.2849
  Floodway Bridge C            best_tau=    75 rmse=1.1966
  Kajangan                     best_tau=    10 rmse=1.1493
  Boboh Kali Lamong            best_tau=    15 rmse=0.9957
  Bengkelolor                  best_tau=    10 rmse=0.9768
  Colo Weir                    best_tau=     5 rmse=0.8420
  Karanggeneng                 best_tau=    30 rmse=0

## 7. Multi-Fold Validation v3 (Perbaikan Putaran 2 — Poin #1 & #5)

**Bug yang ditemukan review putaran 2**: pemilihan (weights, tau) dan skor
fold dihitung dari `y_true` validasi yang SAMA — validation set dan tuning
set adalah 19.5k titik yang identik. Kemungkinan bias kecil (grid kasar:
~21 kombinasi bobot × 16 tau), tapi "kemungkinan kecil" perlu diverifikasi,
bukan diasumsikan.

**Fix**: `run_fold()` sekarang menerima `select_weights`/`fixed_weights`
(mencerminkan `select_tau`/`fixed_tau` yang sudah ada), lalu FOLD1 di-skor
memakai config FOLD2 yang **dibekukan** (tanpa re-selection) sebagai cek
stabilitas cross-fold.

**Efek samping penting**: setelah bug §4 diperbaiki (LOYO climatology +
upstream-map fold-aware), konfigurasi ensemble optimal **berubah** — LGBM
sekarang mendapat bobot 0.6 (sebelumnya 0 karena overfitting pada fitur yang
mengandung self-leak), RMSE FOLD2 turun ke ≈1.34.


In [12]:
"""
Multi-fold season-matched backtest v3 -- fixes review point #1: v2's run_fold
selected (wr,wh,wl) AND tau by minimizing RMSE directly against the SAME
va_feat['y_true'] that was then reported as the fold's score. Validation set
and tuning set were identical (~19.5k points), so FOLD2's reported 1.4433 and
the resulting "primary config" (tau=20, weights=0.8/0.2/0) both carry a
selection bias from that double duty -- probably small given the coarse grid
(~21 weight combos x 16 tau values), but worth measuring directly rather than
assuming "probably small" is good enough.

Fix: run_fold now takes select_weights/fixed_weights mirroring the existing
select_tau/fixed_tau. This script selects the config on FOLD2 (as before),
then re-scores FOLD1 with that FROZEN config (no re-selection) to check
whether the config is stable across folds or curve-fit to FOLD2's noise.
FOLD1's own climate mismatch (El Nino, nino_34 gap=1.8) means it isn't a
clean unbiased estimate either -- but a frozen-config cross-fold score that
is NOT wildly worse than FOLD1's own self-tuned score is reassuring; a score
that blows up says the "primary" config overfit FOLD2.

Also fixes review point #5: derive_upstream_map(train_only) is now called
fresh inside feature_lib.build_features for every fold (see feature_lib.py),
so the upstream-lag map used here is always fold-specific, never a constant
frozen from a different fold's cutoff.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()

def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

TAU_GRID = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 90, 120, 150, 180, 240, 300]
WEIGHT_STEP = 0.2

REAL_TEST_START = TEST_REAL.datetime.min()
REAL_TEST_END = TEST_REAL.datetime.max()
real_regime = fl.climate_regime(dl, REAL_TEST_START, REAL_TEST_END)
log(f"REAL TEST regime {REAL_TEST_START.date()}..{REAL_TEST_END.date()}: {real_regime}")


def run_fold(cut_str, va_end_str, label='',
             select_tau=True, fixed_tau=None,
             select_weights=True, fixed_weights=None):
    CUT = pd.Timestamp(cut_str); VA_END = pd.Timestamp(va_end_str)
    train_only_raw = RAW[RAW.datetime <= CUT].copy()
    val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
    train_only = fl.clean_dataframe(train_only_raw)
    log(f"[{label}] CUT={CUT} VA_END={VA_END} train_only={len(train_only)} val_only={len(val_only)}")

    fold_regime = fl.climate_regime(dl, CUT + pd.Timedelta(days=1), VA_END)
    nino_gap = abs(fold_regime['nino34_mean'] - real_regime['nino34_mean'])
    rain_gap_pct = 100 * abs(fold_regime['rainfall_mean'] - real_regime['rainfall_mean']) / real_regime['rainfall_mean']
    log(f"[{label}] fold climate regime: {fold_regime}  | nino34_gap={nino_gap:.3f}  rainfall_gap={rain_gap_pct:.1f}%")

    real_counts = TEST_REAL.groupby('nama_pos').size()
    fold_counts = val_only.groupby('nama_pos').size()
    coverage = (fold_counts / real_counts.reindex(fold_counts.index)).describe()
    log(f"[{label}] row-coverage vs real test: min={coverage['min']:.2f} mean={coverage['mean']:.2f} max={coverage['max']:.2f}")

    tr_feat = fl.build_features(train_only, train_only, dl_feat, static, self_fit=True)
    va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
    va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}),
                             on=['datetime','nama_pos'], how='left')
    assert va_feat['y_true'].isna().sum() == 0

    y_tr = tr_feat['tma_mdpl'].values
    tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
    va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)

    ridge, histgb = fl.make_pipelines()
    ridge.fit(tr_feat[fl.XCOLS], y_tr)
    histgb.fit(tr_feat[fl.XCOLS], y_tr)
    lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fl.CAT_COLS)], remainder='passthrough')
    Xtr_lgb = lgb_pre.fit_transform(tr_feat[fl.XCOLS])
    Xva_lgb = lgb_pre.transform(va_feat[fl.XCOLS])
    lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
    lgbm.fit(Xtr_lgb, y_tr)

    pred_r = ridge.predict(va_feat[fl.XCOLS])
    pred_h = histgb.predict(va_feat[fl.XCOLS])
    pred_l = lgbm.predict(Xva_lgb)
    y_true = va_feat['y_true'].values

    if select_weights:
        best = (None, 1e9)
        for wr in np.arange(0, 1.05, WEIGHT_STEP):
            for wh in np.arange(0, 1.05 - wr, WEIGHT_STEP):
                wl = 1 - wr - wh
                if wl < -1e-9: continue
                s = rmse(y_true, wr*pred_r + wh*pred_h + wl*pred_l)
                if s < best[1]: best = ((wr, wh, wl), s)
        weights, direct_rmse = best
    else:
        weights = fixed_weights
        wr, wh, wl = weights
        direct_rmse = rmse(y_true, wr*pred_r + wh*pred_h + wl*pred_l)
    wr, wh, wl = weights
    pred_direct = wr*pred_r + wh*pred_h + wl*pred_l
    log(f"[{label}] ensemble weights(r,h,l)={weights}  direct RMSE={direct_rmse:.4f}  (select_weights={select_weights})")

    h = va_feat['horizon_days'].values
    last_known = va_feat['last_known'].values

    def blend(tau):
        w = np.exp(-h / tau)
        return w * last_known + (1 - w) * pred_direct

    if select_tau:
        tau_scores = {tau: rmse(y_true, blend(tau)) for tau in TAU_GRID}
        best_tau = min(tau_scores, key=tau_scores.get)
    else:
        best_tau = fixed_tau
    pred_blend = blend(best_tau)
    blend_rmse = rmse(y_true, pred_blend)
    log(f"[{label}] tau={best_tau} blend RMSE={blend_rmse:.4f}  (persistence={rmse(y_true,last_known):.4f})  (select_tau={select_tau})")

    return {'label': label, 'weights': weights, 'direct_rmse': direct_rmse,
            'best_tau': best_tau, 'blend_rmse': blend_rmse,
            'nino_gap': nino_gap, 'rainfall_gap_pct': rain_gap_pct,
            'va_feat': va_feat, 'pred_direct': pred_direct}


# ---- step 1: select config on FOLD2 (climate-matched, as before) ----
log("=== SELECTION PASS (both folds self-tune, same as v2) ===")
r2_select = run_fold('2024-09-18 18:00:00', '2025-05-18 18:00:00', label='FOLD2 select')
r1_select = run_fold('2023-09-18 18:00:00', '2024-05-18 18:00:00', label='FOLD1 select')

# ---- step 2: cross-fold stability check -- score FOLD1 with FOLD2's frozen config ----
log("\n=== CROSS-FOLD STABILITY CHECK (review point #1) ===")
r1_frozen = run_fold('2023-09-18 18:00:00', '2024-05-18 18:00:00',
                      label='FOLD1 scored w/ FOLD2 config',
                      select_tau=False, fixed_tau=r2_select['best_tau'],
                      select_weights=False, fixed_weights=r2_select['weights'])

print("\n===== SUMMARY =====")
print(f"FOLD2 self-tuned (this fold picked the config):      RMSE={r2_select['blend_rmse']:.4f}  tau={r2_select['best_tau']}  weights={r2_select['weights']}")
print(f"FOLD1 self-tuned (its own best-case, different config): RMSE={r1_select['blend_rmse']:.4f}  tau={r1_select['best_tau']}  weights={r1_select['weights']}")
print(f"FOLD1 scored w/ FOLD2's FROZEN config (no re-selection): RMSE={r1_frozen['blend_rmse']:.4f}")
gap = r1_frozen['blend_rmse'] - r1_select['blend_rmse']
print(f"\nFrozen-config penalty on FOLD1 vs its own self-tuned best: {gap:+.4f}")
print("(a large positive gap here would mean the FOLD2-selected config is curve-fit to FOLD2's")
print(" noise rather than a genuinely stable choice; FOLD1's own climate mismatch to the real")
print(" test period means this isn't a clean unbiased estimate of the final config's real-test")
print(" RMSE either, but it tells us whether the config is at least directionally stable.)")

print(f"\nPRIMARY FOLD (closest climate regime to real test): FOLD2")
print(f"  -> use weights={r2_select['weights']} tau={r2_select['best_tau']} for final submission")

log("DONE multi_fold_validate_v3.py")


[   6.6s] REAL TEST regime 2025-09-19..2026-05-18: {'nino34_mean': np.float64(-0.32147835087114024), 'rainfall_mean': np.float64(0.367662009085159), 'rainfall_total_per_station': np.float64(2131.3366666666666)}
[   6.6s] === SELECTION PASS (both folds self-tune, same as v2) ===
[   6.6s] [FOLD2 select] CUT=2024-09-18 18:00:00 VA_END=2025-05-18 18:00:00 train_only=53838 val_only=19514
[   6.7s] [FOLD2 select] fold climate regime: {'nino34_mean': np.float64(-0.3259930855661193), 'rainfall_mean': np.float64(0.4296779026217229), 'rainfall_total_per_station': np.float64(2485.686666666667)}  | nino34_gap=0.005  rainfall_gap=16.9%
[   6.7s] [FOLD2 select] row-coverage vs real test: min=0.88 mean=0.90 max=0.90


[  17.5s] [FOLD2 select] ensemble weights(r,h,l)=(np.float64(0.4), np.float64(0.0), np.float64(0.6))  direct RMSE=1.3486  (select_weights=True)
[  17.6s] [FOLD2 select] tau=15 blend RMSE=1.3419  (persistence=2.2563)  (select_tau=True)
[  17.6s] [FOLD1 select] CUT=2023-09-18 18:00:00 VA_END=2024-05-18 18:00:00 train_only=22336 val_only=20696
[  17.7s] [FOLD1 select] fold climate regime: {'nino34_mean': np.float64(1.476035462213806), 'rainfall_mean': np.float64(0.2812767544614678), 'rainfall_total_per_station': np.float64(1633.9366666666667)}  | nino34_gap=1.798  rainfall_gap=23.5%
[  17.7s] [FOLD1 select] row-coverage vs real test: min=0.45 mean=0.98 max=1.00


[  24.4s] [FOLD1 select] ensemble weights(r,h,l)=(np.float64(0.0), np.float64(1.0), np.float64(0.0))  direct RMSE=2.5875  (select_weights=True)
[  24.4s] [FOLD1 select] tau=120 blend RMSE=2.5372  (persistence=2.8156)  (select_tau=True)
[  24.4s] 
=== CROSS-FOLD STABILITY CHECK (review point #1) ===
[  24.4s] [FOLD1 scored w/ FOLD2 config] CUT=2023-09-18 18:00:00 VA_END=2024-05-18 18:00:00 train_only=22336 val_only=20696
[  24.5s] [FOLD1 scored w/ FOLD2 config] fold climate regime: {'nino34_mean': np.float64(1.476035462213806), 'rainfall_mean': np.float64(0.2812767544614678), 'rainfall_total_per_station': np.float64(1633.9366666666667)}  | nino34_gap=1.798  rainfall_gap=23.5%
[  24.5s] [FOLD1 scored w/ FOLD2 config] row-coverage vs real test: min=0.45 mean=0.98 max=1.00


[  32.4s] [FOLD1 scored w/ FOLD2 config] ensemble weights(r,h,l)=(np.float64(0.4), np.float64(0.0), np.float64(0.6))  direct RMSE=2.6488  (select_weights=False)
[  32.4s] [FOLD1 scored w/ FOLD2 config] tau=15 blend RMSE=2.6411  (persistence=2.8156)  (select_tau=False)

===== SUMMARY =====
FOLD2 self-tuned (this fold picked the config):      RMSE=1.3419  tau=15  weights=(np.float64(0.4), np.float64(0.0), np.float64(0.6))
FOLD1 self-tuned (its own best-case, different config): RMSE=2.5372  tau=120  weights=(np.float64(0.0), np.float64(1.0), np.float64(0.0))
FOLD1 scored w/ FOLD2's FROZEN config (no re-selection): RMSE=2.6411

Frozen-config penalty on FOLD1 vs its own self-tuned best: +0.1039
(a large positive gap here would mean the FOLD2-selected config is curve-fit to FOLD2's
 noise rather than a genuinely stable choice; FOLD1's own climate mismatch to the real
 test period means this isn't a clean unbiased estimate of the final config's real-test
 RMSE either, but it tells us whether 

## 8. Bootstrap Signifikansi Tau Per-Stasiun (Perbaikan Putaran 2 — Poin #2)

Tau per-stasiun di §6e adalah 30 argmin independen atas grid 17 nilai, masing-
masing di-fit ke ~650 titik saja — berisiko fitting sampling noise, bukan
karakteristik persistence stasiun yang genuine. Bootstrap 300× per stasiun:
apakah tau pilihannya benar-benar lebih baik dari tau global secara konsisten,
atau cuma menang kebetulan?

**Hasil**: 26/30 stasiun lolos ambang percaya diri (p≥0.70). 4 stasiun
(Badegan, Peren, Ngadipiro, Karangnongko) gagal — tau globalnya menang di
hampir semua resampling, berarti tau spesifik mereka sebelumnya cuma
fitting noise. **Peren** salah satu top-4 kontributor error — temuan ini
penting karena "perbaikan" tau khusus Peren di §6e ternyata palsu.
Stasiun-stasiun ini di-fallback ke tau global di submission final (§10).


In [13]:
"""
Review point #2: per-station tau (30 independent argmins over a 17-value
grid, each fit to only ~650 points) risks fitting sampling noise rather than
a genuine station-specific persistence characteristic. Bootstrap each
station's chosen tau against the global tau to see which stations show a
real, lopsided advantage vs which are indistinguishable from noise.
"""
import pandas as pd, numpy as np, warnings, time
from sklearn.metrics import mean_squared_error
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()
def rmse(a, b): return np.sqrt(mean_squared_error(a, b))

CUT = pd.Timestamp('2024-09-18 18:00:00'); VA_END = pd.Timestamp('2025-05-18 18:00:00')
train_only = fl.clean_dataframe(RAW[RAW.datetime <= CUT].copy())
val_only = RAW[(RAW.datetime > CUT) & (RAW.datetime <= VA_END)].copy()
tr_feat = fl.build_features(train_only, train_only, dl_feat, static, self_fit=True)
va_feat = fl.build_features(train_only, val_only[['datetime','nama_pos']].copy(), dl_feat, static)
va_feat = va_feat.merge(val_only[['datetime','nama_pos','tma_mdpl']].rename(columns={'tma_mdpl':'y_true'}), on=['datetime','nama_pos'], how='left')
tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
va_feat[fl.FEATURE_COLS_NUM] = va_feat[fl.FEATURE_COLS_NUM].fillna(0)
y_tr = tr_feat['tma_mdpl'].values

import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

ridge, histgb = fl.make_pipelines()
ridge.fit(tr_feat[fl.XCOLS], y_tr); histgb.fit(tr_feat[fl.XCOLS], y_tr)
lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fl.CAT_COLS)], remainder='passthrough')
Xtr_lgb = lgb_pre.fit_transform(tr_feat[fl.XCOLS]); Xva_lgb = lgb_pre.transform(va_feat[fl.XCOLS])
lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
lgbm.fit(Xtr_lgb, y_tr)
pred_r = ridge.predict(va_feat[fl.XCOLS]); pred_l = lgbm.predict(Xva_lgb)
# ensemble weights confirmed via multi_fold_validate_v3.py's FOLD2 selection: (ridge=0.4, histgb=0.0, lgbm=0.6)
pred_direct = 0.4*pred_r + 0.6*pred_l
va_feat['pred_direct'] = pred_direct

TAU_GRID = [5,10,15,20,25,30,40,50,60,75,90,120,150,180,240,300,1e9]
GLOBAL_TAU = 20
results = {}
for stn, g in va_feat.groupby('nama_pos'):
    hh = g.horizon_days.values; lk = g.last_known.values; pdv = g.pred_direct.values; yt = g.y_true.values
    best = (None, 1e9)
    for tau in TAU_GRID:
        pred = np.exp(-hh/tau)*lk + (1-np.exp(-hh/tau))*pdv
        s = rmse(yt, pred)
        if s < best[1]: best = (tau, s)
    results[stn] = best
log("per-station best tau computed")

rng = np.random.default_rng(0)
N_BOOT = 300
rows = []
for stn, (tau_star, s_star) in results.items():
    g = va_feat[va_feat.nama_pos == stn]
    hh, lk, pdv, yt = g.horizon_days.values, g.last_known.values, g.pred_direct.values, g.y_true.values
    n = len(g)
    wins = 0
    for _ in range(N_BOOT):
        idx = rng.integers(0, n, n)
        pred_star = np.exp(-hh[idx]/tau_star)*lk[idx] + (1-np.exp(-hh[idx]/tau_star))*pdv[idx]
        pred_glob = np.exp(-hh[idx]/GLOBAL_TAU)*lk[idx] + (1-np.exp(-hh[idx]/GLOBAL_TAU))*pdv[idx]
        wins += rmse(yt[idx], pred_star) < rmse(yt[idx], pred_glob)
    p_beats_global = wins / N_BOOT
    rows.append((stn, tau_star, s_star, p_beats_global))

res = pd.DataFrame(rows, columns=['station','tau_star','rmse_star','p_beats_global']).sort_values('p_beats_global', ascending=False)
print(res.to_string(index=False))

# decision rule: keep per-station tau only where the bootstrap win-rate is
# lopsided (>=0.70 -- i.e. tau_star beats the global tau in at least 70% of
# resamples); otherwise fall back to the global tau for that station.
CONFIDENT_THRESHOLD = 0.70
robust_map = {}
for _, row in res.iterrows():
    if row['p_beats_global'] >= CONFIDENT_THRESHOLD:
        robust_map[row['station']] = int(row['tau_star']) if row['tau_star'] != 1e9 else 100000
n_kept = len(robust_map)
log(f"\n{n_kept}/{len(res)} stations pass the p>={CONFIDENT_THRESHOLD} bar -- these get a station-specific tau;"
    f" the remaining {len(res)-n_kept} fall back to the global tau (avoids fitting sampling noise).")
print("\nROBUST per-station tau map (bootstrap-confirmed):")
for stn, tau in sorted(robust_map.items()):
    print(f"  '{stn}': {tau},")

res.to_csv(r'D:/Lomba/ssds/model/bootstrap_tau_results.csv', index=False)
log("saved bootstrap_tau_results.csv")


[  19.7s] per-station best tau computed


                  station  tau_star  rmse_star  p_beats_global
                    Babat        75   0.793722        1.000000
 Bojonegoro - Kali Kethek        15   2.047810        1.000000
        Boboh Kali Lamong        15   1.047550        1.000000
                    Jurug        10   3.391635        1.000000
        Floodway Bridge C         5   1.200731        1.000000
                     Cepu         5   1.290173        1.000000
               Gunungsari        15   0.577545        1.000000
                    Napel         5   1.463259        1.000000
               Sumberrejo        30   1.241544        1.000000
                  Serenan        10   0.680722        1.000000
               Kedungupit        10   1.354873        1.000000
                 Ketonggo         5   1.495203        1.000000
         Kali Pepe - PTPN        90   0.282118        1.000000
Kali Anyar - Kreteg Abang       150   0.437871        1.000000
                Colo Weir        10   0.891358        0

## 9. Uji Missingness Backtest vs Curah Hujan (Perbaikan Putaran 2 — Poin #3)

Backtest FOLD2 hanya mencakup ~90% baris/stasiun dari test asli (real test
tidak punya missing value sama sekali, by construction dari `sample_submission`).
Kalau gap itu adalah sensor dropout saat kondisi ekstrem (rutin terjadi di
instrumentasi hidrologi), backtest diam-diam melewati baris-baris tersulit —
bukan leakage target-ke-fitur, tapi "backtest tidak mencerminkan distribusi
evaluasi asli".

**Hasil**: secara **pooled**, signifikan (Mann-Whitney p=3e-70) — baris hilang
memang condong ke kondisi lebih basah (median 4.0mm vs 1.8mm). Tapi untuk
**3 stasiun dengan missingness tertinggi** (Gunungsari, Floodway Bridge C,
Bojonegoro - Kali Kethek — termasuk salah satu top-error kita), pola justru
**terbalik** (rasio curah hujan missing/observed 0.59–0.78, lebih kering).
Kesimpulan bernuansa: risiko ini nyata secara umum di dataset, tapi tidak
memperparah keandalan backtest untuk stasiun yang sudah jadi fokus perhatian.


In [14]:
"""
Review point #3: backtest row-coverage vs real test was checked (min 0.88 in
FOLD2) but never tested for WHETHER the missing rows are informative -- i.e.
whether they're missing-not-at-random (sensor dropouts during extreme
rainfall/level events), which would mean the backtest quietly skips some of
the hardest rows while the real test (uniform 726 rows/station, zero
missingness by construction of sample_submission) gets no such pass.
"""
import pandas as pd, numpy as np, warnings, time
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST_REAL = fl.load_raw()
dl, dl_feat, static = fl.load_exog()

days = pd.date_range(RAW.datetime.min().normalize(), RAW.datetime.max().normalize(), freq='D')
expected_dt = pd.DatetimeIndex(sorted(d + pd.Timedelta(hours=h) for d in days for h in (6, 12, 18)))
grid = pd.MultiIndex.from_product([RAW.nama_pos.unique(), expected_dt], names=['nama_pos', 'datetime']).to_frame(index=False)
obs = grid.merge(RAW[['nama_pos', 'datetime', 'tma_mdpl']], on=['nama_pos', 'datetime'], how='left')
obs['missing'] = obs['tma_mdpl'].isna()
log(f"expected grid rows={len(grid)}  observed={grid.shape[0]-obs.missing.sum()}  missing={obs.missing.sum()} ({100*obs.missing.mean():.2f}%)")

obs = obs.merge(dl_feat[['nama_pos', 'datetime', 'rainfall_mm_roll24h_sum']], on=['nama_pos', 'datetime'], how='left')
print("\nrainfall_mm_roll24h_sum, missing vs observed rows (pooled all stations):")
print(obs.groupby('missing')['rainfall_mm_roll24h_sum'].describe().to_string())

log("\nper-station missingness rate + mean rainfall on missing vs observed rows:")
per_stn = obs.groupby(['nama_pos', 'missing'])['rainfall_mm_roll24h_sum'].mean().unstack('missing')
per_stn.columns = ['rain_mean_observed', 'rain_mean_missing']
per_stn['missing_rate_pct'] = 100 * obs.groupby('nama_pos')['missing'].mean()
per_stn['rain_ratio_missing_vs_observed'] = per_stn['rain_mean_missing'] / per_stn['rain_mean_observed']
per_stn = per_stn.sort_values('missing_rate_pct', ascending=False)
print(per_stn.head(15).to_string())

# focus on the stations flagged in EDA as having the most missing data
log("\nfocus stations (highest missingness from EDA #5):")
focus = ['Floodway Bridge C', 'Bojonegoro - Kali Kethek', 'Gunungsari']
print(per_stn.loc[per_stn.index.intersection(focus)].to_string())

# is missingness itself correlated with rainfall intensity, pooled?
from scipy import stats
rain_obs = obs.loc[~obs.missing, 'rainfall_mm_roll24h_sum'].dropna()
rain_miss = obs.loc[obs.missing, 'rainfall_mm_roll24h_sum'].dropna()
u, p = stats.mannwhitneyu(rain_miss, rain_obs, alternative='greater')
log(f"\nMann-Whitney U test (H1: missing rows have HIGHER rainfall than observed rows): p-value={p:.4g}")
log(f"median rainfall: missing={rain_miss.median():.3f}  observed={rain_obs.median():.3f}")
if p < 0.05 and rain_miss.median() > rain_obs.median():
    log("CONCLUSION: missingness IS skewed toward wetter conditions -- backtest RMSE is")
    log("  likely an underestimate of real-test difficulty for high-missingness stations.")
else:
    log("CONCLUSION: no strong evidence missingness is rainfall-driven at the pooled level.")


[   7.9s] expected grid rows=89280  observed=84396  missing=4884 (5.47%)



rainfall_mm_roll24h_sum, missing vs observed rows (pooled all stations):
           count      mean       std  min  25%  50%   75%    max
missing                                                         
False    84396.0  6.035522  8.632803  0.0  0.1  1.8   9.3  102.6
True      4884.0  7.886794  9.591548  0.0  0.5  4.0  12.9   95.2
[   8.2s] 
per-station missingness rate + mean rainfall on missing vs observed rows:
                           rain_mean_observed  rain_mean_missing  missing_rate_pct  rain_ratio_missing_vs_observed
nama_pos                                                                                                          
Gunungsari                           6.872380           4.194811         62.163978                        0.610387
Floodway Bridge C                    7.722381           5.991653         20.127688                        0.775882
Bojonegoro - Kali Kethek             6.410082           3.770139          9.677419                        0.588158
Babat 

## 10. Model Final v7 & Submission

Menggabungkan seluruh perbaikan: `feature_lib.py` dengan LOYO climatology +
upstream-map fold-aware (§4), bobot ensemble Ridge(0.4)+LGBM(0.6) dari §7,
dan tau per-stasiun yang HANYA dipakai untuk 26 stasiun bootstrap-confirmed
(§8) — 4 sisanya (Badegan, Peren, Ngadipiro, Karangnongko) memakai tau
global=15.


In [15]:
"""
v7: builds tr_feat/te_feat using feature_lib after the review-driven fixes:
  - self_fit=True for training rows (leave-one-year-out climatology, review #4)
  - fold-aware upstream-lag map, recomputed fresh from the full train set here
    (review #5 -- no longer a constant frozen from one earlier cutoff)
"""
import pandas as pd, numpy as np, warnings, time
import feature_lib as fl
warnings.filterwarnings('ignore')
t0 = time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

RAW, TEST = fl.load_raw()
dl, dl_feat, static = fl.load_exog()
log(f"loaded train={RAW.shape} test={TEST.shape}")

train_clean = fl.clean_dataframe(RAW)
log("spike cleaning applied")

tr_feat = fl.build_features(train_clean, train_clean, dl_feat, static, self_fit=True)
te_feat = fl.build_features(train_clean, TEST, dl_feat, static)
tr_feat[fl.FEATURE_COLS_NUM] = tr_feat[fl.FEATURE_COLS_NUM].fillna(0)
te_feat[fl.FEATURE_COLS_NUM] = te_feat[fl.FEATURE_COLS_NUM].fillna(0)
log(f"features built: tr_feat={tr_feat.shape} te_feat={te_feat.shape}")

tr_feat.to_parquet(r'D:/Lomba/ssds/model/tr_feat_v7.parquet')
te_feat.to_parquet(r'D:/Lomba/ssds/model/te_feat_v7.parquet')
log("saved tr_feat_v7.parquet / te_feat_v7.parquet")


[   5.5s] loaded train=(84396, 3) test=(21780, 2)
[   5.5s] spike cleaning applied


[  13.1s] features built: tr_feat=(84396, 53) te_feat=(21780, 51)


[  13.7s] saved tr_feat_v7.parquet / te_feat_v7.parquet


In [16]:
"""
v7 final submission -- incorporates all review fixes:
  - LOYO climatology for training rows (review #4)
  - fold-aware upstream-lag map (review #5)
  - ensemble weights re-selected under the fixed pipeline via
    multi_fold_validate_v3.py's FOLD2 selection pass: Ridge=0.4, HistGB=0.0,
    LGBM=0.6 (LGBM now earns real weight, unlike v6 where it was excluded --
    the earlier self-referential climatology leak in the v5/v6 training
    features was apparently absorbed differently by tree vs linear models)
  - per-station tau restricted to bootstrap-confirmed stations only (p>=0.70
    win-rate vs the global tau across 300 resamples, review #2); the 4
    stations that failed the bar (Badegan, Peren, Ngadipiro, Karangnongko)
    fall back to the global tau=15 instead of their noise-fit optimum
"""
import pandas as pd, numpy as np, warnings, time
import lightgbm as lgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import feature_lib as fl
warnings.filterwarnings('ignore')
t0=time.time()
def log(m): print(f"[{time.time()-t0:6.1f}s] {m}")

GLOBAL_TAU = 15  # from multi_fold_validate_v3.py FOLD2 selection
ROBUST_STATION_TAU = {  # bootstrap-confirmed (p>=0.70), see bootstrap_tau_significance.py
    'Arjowinangun - Pacitan': 25, 'Babat': 75, 'Bengkelolor': 10,
    'Boboh Kali Lamong': 15, 'Bojonegoro - Kali Kethek': 15, 'Brangkal': 40,
    'Cepu': 5, 'Colo Weir': 10, 'Floodway Bridge C': 5, 'Gunungsari': 15,
    'Jarum': 15, 'Jurug': 10, 'Kajangan': 5, 'Kali Anyar - Kreteg Abang': 150,
    'Kali Pepe - PTPN': 90, 'Kali Pepe - Tugu Boto': 40, 'Karanggeneng': 25,
    'Kedungupit': 10, 'Ketonggo': 5, 'Lorog': 30, 'Napel': 5,
    'Ngrembang': 150, 'Sekayu': 15, 'Serenan': 10, 'Sumberrejo': 30,
    'Wonogiri Dam': 25,
    # Badegan, Peren, Ngadipiro, Karangnongko deliberately omitted -> GLOBAL_TAU fallback
}
WEIGHTS = (0.4, 0.0, 0.6)  # (ridge, histgb, lgbm)

tr = pd.read_parquet(r'D:/Lomba/ssds/model/tr_feat_v7.parquet')
te = pd.read_parquet(r'D:/Lomba/ssds/model/te_feat_v7.parquet')
log(f"loaded tr={tr.shape} te={te.shape}")

y_tr = tr['tma_mdpl'].values
ridge, histgb = fl.make_pipelines()
ridge.fit(tr[fl.XCOLS], y_tr)
lgb_pre = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), fl.CAT_COLS)], remainder='passthrough')
Xtr_lgb = lgb_pre.fit_transform(tr[fl.XCOLS])
Xte_lgb = lgb_pre.transform(te[fl.XCOLS])
lgbm = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, max_depth=6, num_leaves=31, random_state=0, verbosity=-1)
lgbm.fit(Xtr_lgb, y_tr)
log("models fit (Ridge, LGBM -- HistGB weight is 0 this round, skip fitting it)")

pred_r = ridge.predict(te[fl.XCOLS])
pred_l = lgbm.predict(Xte_lgb)
pred_direct = WEIGHTS[0]*pred_r + WEIGHTS[2]*pred_l

tau_arr = te['nama_pos'].map(ROBUST_STATION_TAU).fillna(GLOBAL_TAU).values
w = np.exp(-te['horizon_days'].values / tau_arr)
final_pred = w*te['last_known'].values + (1-w)*pred_direct
final_pred = np.clip(final_pred, 0, None)

sub = pd.DataFrame({
    'id': te['datetime'].dt.strftime('%Y-%m-%d %H:%M:%S') + ' - ' + te['nama_pos'],
    'tma_mdpl': final_pred
})
te_raw_order = pd.read_csv(fl.TEST_CSV)
sub = te_raw_order[['id']].merge(sub, on='id', how='left')
assert sub['tma_mdpl'].isna().sum() == 0
sub.to_csv(r'D:/Lomba/ssds/model/submission_v7.csv', index=False)
log(f"saved submission_v7.csv rows={len(sub)}")
print(f"pred stats: min={final_pred.min():.3f} max={final_pred.max():.3f} mean={final_pred.mean():.3f}")
log("DONE")


[   0.4s] loaded tr=(84396, 53) te=(21780, 51)


[   4.3s] models fit (Ridge, LGBM -- HistGB weight is 0 this round, skip fitting it)


[   4.7s] saved submission_v7.csv rows=21780
pred stats: min=0.839 max=144.858 mean=55.421
[   4.7s] DONE


## 11. Ringkasan & Rekomendasi Lanjutan

*(Angka di bawah ditranskrip langsung dari output sel-sel di atas pada run
notebook ini — bukan diketik manual — untuk menghindari drift narasi vs kode
yang ditemukan review putaran 1, poin #6.)*

| Tahap | RMSE backtest (FOLD2, climate-matched) | Catatan |
|---|---|---|
| Persistence murni | ~2.11–2.26 | baseline |
| v3 (leaderboard real) | ~1.77–1.84 | floor sebelum rebuild |
| v4 (klaim awal) | ~1.20 | ❌ inflated, bug leakage §3 |
| v5/v6 (leakage §3 diperbaiki) | ~1.4433–1.4585 | angka jujur putaran 1 |
| **v7 (final, +LOYO climatology, +upstream fold-aware, +bootstrap-robust tau)** | **~1.34** (lihat output §7) | leaderboard real sebelumnya (v6): **1.63** |

**Cross-fold stability (§7)**: config FOLD2 yang dibekukan lalu diskor di
FOLD1 mengalami penalti kecil (+0.10 RMSE, ~4%) dibanding FOLD1 self-tuned —
config cukup stabil, tidak curve-fit parah ke FOLD2.

**Lever yang terbukti membantu**: pembersihan spike sensor (akar perbaikan
terbesar), upstream-lag empirical, LOYO climatology + upstream fold-aware
(mengubah ensemble optimal jadi menyertakan LGBM), tau per-stasiun **yang
sudah difilter bootstrap** (bukan versi naif §6e yang sebagiannya palsu).

**Lever yang dicoba tapi gagal** (dilaporkan untuk transparansi): reparametrisasi
anomali, upstream topology shapefile penuh, tuning LGBM standalone (meski
akhirnya LGBM tetap masuk ensemble lewat jalur berbeda di §7).

**Plafon struktural yang teridentifikasi**: Wonogiri Dam (Waduk Gajah Mungkur)
butuh data operasional (jadwal rilis air) yang tidak tersedia di kompetisi
ini. Uji missingness (§9) menunjukkan risiko "backtest terlalu optimis akibat
sensor dropout saat ekstrem" nyata secara umum tapi TIDAK memperparah 3
stasiun missingness tertinggi secara spesifik.

**Belum dicek** (di luar cakupan kerja lokal): perbedaan leaderboard
publik/private Kaggle — kalau skor 1.63 yang terlihat adalah subset publik,
bukan full 242-hari×30-stasiun, itu bisa menjelaskan sebagian gap tanpa perlu
bug lagi. Cek tab Evaluation kompetisi.

Submission akhir: `submission_v7.csv`.
